# EmPath: clip 2021 Texas canopy in Google Colab
Upload this notebook at https://colab.research.google.com/ using **File → Upload notebook**. Run the cells in order on a CPU runtime. The ~2.25 GB download and extraction happen on Colab, not your laptop. Allow at least 30 GB free runtime disk space; download speed depends on Zenodo and Colab. Runtime files disappear when the session resets.

Scope: **75025, 75024, 75074, and comparison 75023**, using the project's saved 2020 Census ZCTA polygons. Default: intersect with **Collin County (48085)**, from the Census 2026 county service. Set `FULL_ZCTAS=True` below to include the full footprints instead. ZCTAs approximate ZIP codes.

Source: [Diamond, Sunde & Elliott, Texas 2021 canopy, 10 m](https://zenodo.org/records/8200761). This is canopy percentage, not time-specific sidewalk shade. The archive includes an `.img` and companion `.ige`; the script extracts both. Only the scope-sized raster window is read into memory. No resampling; outside-scope and source NoData pixels remain masked. Zero canopy remains valid.

This notebook embeds the project script and boundary snapshot, so no repository upload or account credentials are needed. It has not yet been run against the full statewide dataset; value checks stop execution if valid pixels fall outside 0–100.

In [ ]:
%pip -q install "geopandas>=1,<2" "rasterio>=1.3,<2" "matplotlib>=3.8,<4" requests "affine>=2.4,<3"

In [ ]:
from pathlib import Path
import shutil
WORK = Path('/content/empath_canopy')
WORK.mkdir(parents=True, exist_ok=True)
print('Free disk (GB):', round(shutil.disk_usage(WORK).free / 1e9, 1))
FULL_ZCTAS = False
ZIPCODES = ['75025', '75024', '75074', '75023']


In [ ]:
BOUNDARY_SNAPSHOT = '{"type": "FeatureCollection", "features": [{"type": "Feature", "geometry": {"type": "Polygon", "coordinates": [[[-96.70982499994072, 33.02474599978979], [-96.70983800035953, 33.023881000103984], [-96.70982099974277, 33.02293699992131], [-96.70981300024516, 33.022802999709526], [-96.70979399997859, 33.0224160000733], [-96.70973399970413, 33.021948000065315], [-96.70963499996994, 33.02139500036841], [-96.70955100030434, 33.020924000169764], [-96.70952899966471, 33.02083699963704], [-96.70951000029645, 33.02073599979399], [-96.70946599991552, 33.020553000365254], [-96.70945599986979, 33.020491000200956], [-96.70932600017345, 33.01996799965567], [-96.7092800001427, 33.01977799969736], [-96.70924900018056, 33.019653999741614], [-96.70922099969316, 33.01955799994766], [-96.70909799966948, 33.019139999814634], [-96.70867000004733, 33.01769899980332], [-96.70860300010015, 33.017475999978366], [-96.70860300010015, 33.01745700021717], [-96.70840499973346, 33.01666399983209], [-96.70831600004499, 33.01604300019429], [-96.70828300043303, 33.015788000142486], [-96.7082390000521, 33.015329000299985], [-96.7082099997398, 33.01484799975562], [-96.70791000016413, 33.011049000053156], [-96.70781100042993, 33.00960499996837], [-96.70778299994252, 33.00926199983442], [-96.70777600026983, 33.00920000001705], [-96.70777200007187, 33.009151999715414], [-96.70776200002612, 33.00905300009371], [-96.70775199998039, 33.00894899986153], [-96.70771699982029, 33.008378000190326], [-96.7076969997288, 33.007959000131486], [-96.70769199970593, 33.00783399971409], [-96.70770900032268, 33.00701899991174], [-96.70771900036843, 33.00674899976775], [-96.7077539996302, 33.0063300002572], [-96.70777000042206, 33.006145999877276], [-96.70780399985891, 33.00588399979584], [-96.70787199963098, 33.00526499971251], [-96.70788599987469, 33.00515400019119], [-96.7079920001799, 33.00431000013778], [-96.70800100040073, 33.004232999813944], [-96.7078940002706, 33.00423499996997], [-96.70773500026193, 33.00423999998313], [-96.70732900038108, 33.00423499996997], [-96.70724599964208, 33.00423599967125], [-96.70705999986926, 33.00423200011256], [-96.70666400003414, 33.00421499991441], [-96.70529900007843, 33.00421600036935], [-96.70488300015185, 33.00418499982265], [-96.7034300003326, 33.00408000017441], [-96.70231200024912, 33.00396699975328], [-96.70101599989042, 33.003861000140034], [-96.70086500027767, 33.003851999776096], [-96.70014100037949, 33.003794000254814], [-96.69968500026991, 33.00375600010615], [-96.69958700036062, 33.00374899989854], [-96.6994679996366, 33.003736999648986], [-96.69934899981092, 33.003730999900306], [-96.69919499982514, 33.003722000276476], [-96.69901900009803, 33.003712000191136], [-96.69727099982771, 33.00361299973706], [-96.69664999986169, 33.003576999681364], [-96.69637899970004, 33.003565000161736], [-96.69542100034843, 33.00351800026982], [-96.69514599998882, 33.003510000333776], [-96.69422100024919, 33.0034530002978], [-96.69151300007913, 33.00326999965589], [-96.68899599970477, 33.00312400011066], [-96.68814999960786, 33.003073000372495], [-96.68550300043547, 33.00294999999646], [-96.68354699975268, 33.002857999719424], [-96.6825209997307, 33.00279500020322], [-96.6824499995856, 33.00279200029684], [-96.68237900033878, 33.00278599973053], [-96.68237500014082, 33.00257000016454], [-96.68236599992001, 33.00241199978434], [-96.6821900001929, 33.00240800014301], [-96.68206200014639, 33.00239700018654], [-96.6820069998948, 33.0023899998711], [-96.68193499992478, 33.00238400003082], [-96.68133400005026, 33.002315000331016], [-96.68048399975537, 33.00219800025825], [-96.67930600029574, 33.00194200037558], [-96.67842199966567, 33.001699999738655], [-96.67772000040712, 33.001457999944684], [-96.67710100009091, 33.00123499973368], [-96.67699599961061, 33.00119400002879], [-96.67657700020928, 33.00102700029903], [-96.67586199963361, 33.00066200036661], [-96.67567000001299, 33.000552000070215], [-96.67502099955955, 33.000182999937195], [-96.67436799980649, 32.99976899964138], [-96.67399799991064, 32.99952500031507], [-96.67340000040915, 32.99912699986103], [-96.67214099986036, 32.99815600005079], [-96.67195099988956, 32.99799499998369], [-96.67166600038253, 32.997718000163395], [-96.67152499991722, 32.99756199968873], [-96.6713530003881, 32.997397000175994], [-96.67124299988491, 32.99727599973452], [-96.67113599975478, 32.99715899976006], [-96.67098799961678, 32.996994000247156], [-96.67082500030847, 32.99698899982332], [-96.67056199964443, 32.99699999970063], [-96.66974499985983, 32.996988000039856], [-96.66914599963512, 32.99699999970063], [-96.66833100039865, 32.99699999970063], [-96.66525499998255, 32.99698300036911], [-96.66513799980669, 32.99698300036911], [-96.66503300022471, 32.99697499984053], [-96.66496000042979, 32.99697499984053], [-96.66496399972942, 32.997056000356004], [-96.66496899975228, 32.99713499972678], [-96.66496799992737, 32.9971969997366], [-96.66496799992737, 32.99726199979727], [-96.66496700010246, 32.997461000311276], [-96.66495399968369, 32.9993519996398], [-96.66493399959218, 33.002723999881326], [-96.66493600014032, 33.002806000110006], [-96.66486699964504, 33.00319599982371], [-96.66484400007882, 33.003619999955404], [-96.66481999978936, 33.004105999984155], [-96.6643150001743, 33.00412500036184], [-96.66431900037226, 33.003930999842055], [-96.6622720003512, 33.00400699997226], [-96.66228599969658, 33.0034549997182], [-96.66191900017377, 33.0034530002978], [-96.65924000031602, 33.00343200035462], [-96.65923799976788, 33.003513000215726], [-96.65923399956993, 33.00372700031872], [-96.65922400042251, 33.00427400036653], [-96.65883100006211, 33.00427000005619], [-96.6549229997928, 33.00423900028184], [-96.65459999975262, 33.00422999995652], [-96.65346499995069, 33.00428899964517], [-96.65325900008638, 33.004300000119095], [-96.65172599990119, 33.00439899980216], [-96.65154599997614, 33.00439699964985], [-96.6507540003059, 33.00447000028263], [-96.64748799991898, 33.0046289996871], [-96.64447900034837, 33.004754999821834], [-96.64178399969876, 33.004860999867546], [-96.63966399988276, 33.00495800028815], [-96.63910700038916, 33.004981000007234], [-96.63871300020386, 33.00499800005776], [-96.63821199988844, 33.00501799993574], [-96.63754099969368, 33.00502400035023], [-96.63701699981206, 33.00502000007397], [-96.63548000032723, 33.0050050001662], [-96.63389299971539, 33.00500899968984], [-96.63331500030539, 33.005010000135776], [-96.63320599962712, 33.00501099982827], [-96.63312899963422, 33.00506299964466], [-96.6329999997628, 33.005125000109985], [-96.6328659998685, 33.00518100012832], [-96.63277299998208, 33.005220999796805], [-96.63264499993556, 33.00529700032227], [-96.63256100026997, 33.005366999638134], [-96.63247199968319, 33.00545400036775], [-96.63245900016273, 33.005491000097926], [-96.63247199968319, 33.00552799981251], [-96.63246199963746, 33.005618999879324], [-96.63244300026919, 33.00571499978187], [-96.63245999998763, 33.005757000083044], [-96.63249599997265, 33.00580499997228], [-96.63251400041429, 33.005819999743956], [-96.63253199995766, 33.005878999875875], [-96.63253199995766, 33.00590299979056], [-96.6325560002471, 33.00594999991844], [-96.6326409997376, 33.00602700025021], [-96.63266300037722, 33.00613100016104], [-96.63270199983698, 33.006216000026114], [-96.63272299975337, 33.006248000290874], [-96.63275000041587, 33.006270999673596], [-96.63282400003571, 33.00627799968102], [-96.63356800002538, 33.00685499967734], [-96.63365699971384, 33.00695999978901], [-96.63370000026985, 33.00704600003134], [-96.6337029997446, 33.007203999644815], [-96.63366699975958, 33.00727500012755], [-96.63360099963734, 33.00735299972194], [-96.6335410002612, 33.00740699974835], [-96.63346199972015, 33.00742699983351], [-96.63329100001594, 33.00743000033556], [-96.63313899967997, 33.00741599974956], [-96.63298100039454, 33.007410000251134], [-96.63283499990636, 33.00743000033556], [-96.63277899982987, 33.00745500024638], [-96.63275600026364, 33.007522000364204], [-96.63277299998208, 33.0075679999659], [-96.63281599963977, 33.00763499999788], [-96.6329010000286, 33.007706999795616], [-96.63298100039454, 33.00776200002002], [-96.63303699957272, 33.00785399970237], [-96.63306000003726, 33.00793900016701], [-96.63306599988503, 33.00798899969303], [-96.63304300031882, 33.008063000024364], [-96.63302700042529, 33.00818600005087], [-96.63300399996076, 33.00828000010855], [-96.63298999971705, 33.0084069999553], [-96.63299700028807, 33.00871600023635], [-96.63298999971705, 33.00886799971893], [-96.63295399973205, 33.008987999680485], [-96.63289500018082, 33.00908500008253], [-96.6328089999671, 33.0091649996271], [-96.63268699976837, 33.009219999695645], [-96.63256100026997, 33.009240000122915], [-96.63236999957596, 33.009219999695645], [-96.63213199992457, 33.00923100030787], [-96.63203300019038, 33.009255999708216], [-96.63198700015963, 33.00931400015796], [-96.63195100017461, 33.00937999997391], [-96.63195100017461, 33.009483000055994], [-96.63196700006814, 33.00957399980901], [-96.63196399969507, 33.00969600034882], [-96.63194100012886, 33.00986999996221], [-96.63192100003738, 33.00998499962649], [-96.63191799966432, 33.01008100005023], [-96.63188400022746, 33.01019800021351], [-96.63185400009023, 33.01041199991744], [-96.63184700041752, 33.01049500016272], [-96.63186099976294, 33.01059500005494], [-96.63191799966432, 33.01083200000208], [-96.63197600028897, 33.01091400022651], [-96.63218499962802, 33.01111399970065], [-96.63244800029207, 33.01129400027086], [-96.63262599966899, 33.0113949996306], [-96.63277700018004, 33.01147399972043], [-96.63302700042529, 33.01158900030637], [-96.63323999996231, 33.011648999962965], [-96.63340499981875, 33.01167400018451], [-96.63371199996541, 33.01167400018451], [-96.63411799984628, 33.01170900033211], [-96.63427799967985, 33.011739000124656], [-96.63441599977212, 33.01174999966945], [-96.63456700028317, 33.0117869997716], [-96.63470100017747, 33.01185699996315], [-96.63479900008676, 33.0119290000807], [-96.63484299956937, 33.01197799999105], [-96.63487899955439, 33.01203400017815], [-96.63492399976023, 33.012205999723264], [-96.63499499990535, 33.01238499996229], [-96.6350479996088, 33.01247799970316], [-96.63511499955595, 33.01255200003436], [-96.6352040001427, 33.012628000269224], [-96.63530700007487, 33.012665000003], [-96.6353999999613, 33.012689999936526], [-96.63550700009142, 33.01268400004348], [-96.63559199958192, 33.01265799974916], [-96.63566299972702, 33.01260799985694], [-96.63601500007954, 33.01226900005086], [-96.63622400031691, 33.012123000335045], [-96.63641099991466, 33.01205300035463], [-96.6364999996031, 33.01203400017815], [-96.63659800041073, 33.01201899995664], [-96.63666500035788, 33.012022000302444], [-96.63673200030503, 33.01203699977015], [-96.63679400022929, 33.01206399986019], [-96.63683400041226, 33.01210099983061], [-96.63687000039727, 33.01214599969232], [-96.63689700016145, 33.01219700021089], [-96.63689700016145, 33.01227299999835], [-96.63687399969692, 33.012323999690175], [-96.63678099981051, 33.01245200010133], [-96.63669600032001, 33.01258299990017], [-96.63660300043358, 33.01271000033016], [-96.63653099956525, 33.01287299986665], [-96.6364910002806, 33.01300799970756], [-96.6364910002806, 33.013075000126605], [-96.63650499962598, 33.013122999799656], [-96.6365759997711, 33.01320199984802], [-96.63666899965752, 33.01326599981629], [-96.63672800010707, 33.01328099982568], [-96.63679400022929, 33.01327800027594], [-96.63711499972135, 33.01322900033462], [-96.63730800006519, 33.01321299996357], [-96.63744600015745, 33.013235999789856], [-96.6377390000604, 33.01334000023954], [-96.6380400003593, 33.01347700006526], [-96.63812300019998, 33.013497999876755], [-96.6382160000864, 33.013501999768515], [-96.6383369995619, 33.01349399998477], [-96.63845999958555, 33.01347199982242], [-96.6387090000059, 33.01337900037654], [-96.63895800042624, 33.013306000337806], [-96.63915099987176, 33.01326900011968], [-96.63937299962961, 33.01326900011968], [-96.63953900020928, 33.01328699967804], [-96.63968200032443, 33.013323000290335], [-96.63986899992217, 33.0133899997167], [-96.63993099984644, 33.01338799976834], [-96.63992300034884, 33.01344100026912], [-96.63994400026525, 33.01349099968903], [-96.63997199985434, 33.01353899988893], [-96.64001799988509, 33.01357799993793], [-96.64013699971078, 33.013626999685265], [-96.64017799971865, 33.01367499981114], [-96.64021200005385, 33.01373999978296], [-96.64021200005385, 33.01383999976193], [-96.640166999848, 33.01392800001123], [-96.64009300022815, 33.01399000027104], [-96.63988399999079, 33.01408999996656], [-96.63980299979993, 33.01414699990791], [-96.63973699967771, 33.01420100029418], [-96.63966200023292, 33.014311000141724], [-96.63964299996636, 33.01444600004095], [-96.63966200023292, 33.01450999985965], [-96.63971800030944, 33.01462000007494], [-96.63975599994427, 33.01468199984825], [-96.63983999960986, 33.0147610000065], [-96.63988300016587, 33.01478000034865], [-96.63994400026525, 33.01478900035069], [-96.63998899957278, 33.01478000034865], [-96.6401319996879, 33.01472199972744], [-96.6402010001832, 33.01470199979014], [-96.64026300010747, 33.01469099986029], [-96.64031900018395, 33.01468800035857], [-96.64036199984167, 33.014712999718554], [-96.64038199993315, 33.01474400033119], [-96.64039500035194, 33.014784999764025], [-96.64040799987242, 33.01490199971255], [-96.64043999965948, 33.01494600007153], [-96.64047700036772, 33.01496900019909], [-96.6405450001398, 33.01498300033927], [-96.64059199999544, 33.01497699984903], [-96.6406740000112, 33.014951999810606], [-96.64074799963106, 33.01494299982521], [-96.6407869999891, 33.01493300025879], [-96.64083200019495, 33.014915999863426], [-96.64089800031718, 33.014897999884305], [-96.64097000028721, 33.014899000217994], [-96.64101499959476, 33.014919000110666], [-96.64105700032586, 33.01495699996951], [-96.64109099976272, 33.0150059997039], [-96.64111199967913, 33.0150570000749], [-96.64112099989995, 33.01509899969364], [-96.64111699970199, 33.01517699990103], [-96.64111900025013, 33.01525100022707], [-96.6411300001208, 33.01531599979045], [-96.64114799966414, 33.01534700019114], [-96.6411870000222, 33.015375000349245], [-96.64124499974851, 33.01538499986553], [-96.64129099977926, 33.015365999653746], [-96.6413929998865, 33.01525399970969], [-96.64144300011523, 33.01521900021583], [-96.64148499994803, 33.015199999968274], [-96.6415810002075, 33.01518799977019], [-96.64167699956865, 33.01519200033856], [-96.6417150001018, 33.015199999968274], [-96.6417519999117, 33.01521299974165], [-96.64178600024691, 33.015233999893006], [-96.64183300010258, 33.01527600018071], [-96.64203700031707, 33.01542000004673], [-96.64224900002918, 33.01552400018033], [-96.64246599976416, 33.01568199967059], [-96.64252500021371, 33.01576000011547], [-96.64257999956698, 33.01586000006344], [-96.6426020002066, 33.01592199971809], [-96.64264999988717, 33.01614299982125], [-96.64267699965134, 33.016201999826684], [-96.64278799997943, 33.016343000241534], [-96.64282099959141, 33.016362000242786], [-96.64289599993447, 33.016457000187714], [-96.64304700044552, 33.016708999624996], [-96.64308400025544, 33.01680399994925], [-96.64308400025544, 33.01689700030238], [-96.64307199966157, 33.016964000026206], [-96.64304499989738, 33.017026999967385], [-96.64299100036904, 33.01708999986361], [-96.64291500020104, 33.01716599995345], [-96.642799999675, 33.01726000023167], [-96.64271600000941, 33.01733900033988], [-96.64260800005438, 33.017430999823674], [-96.6425389995591, 33.01746599994596], [-96.64248699968054, 33.01746499964139], [-96.64237899972551, 33.01741000019558], [-96.64228099981624, 33.01738300023596], [-96.64223899998342, 33.017378999767], [-96.64207599977682, 33.01738699995155], [-96.64190300042277, 33.01742200009131], [-96.64176100013255, 33.0174910000248], [-96.6417309999953, 33.01752799997784], [-96.64170900025402, 33.01756100020785], [-96.6416940001854, 33.01764000004636], [-96.64169300036048, 33.01770199995588], [-96.64169800038336, 33.01777800027145], [-96.64174500023904, 33.01788800017609], [-96.64178700007183, 33.017938999633365], [-96.64185900004183, 33.01801200035598], [-96.64194699990537, 33.01803999966779], [-96.64199399976106, 33.018080999827745], [-96.64201999970032, 33.01812199996862], [-96.64203300011911, 33.01817499988368], [-96.64203399994403, 33.01821599998086], [-96.64201399985252, 33.01826099973484], [-96.64190300042277, 33.01838300030825], [-96.64185300019406, 33.01846399999681], [-96.64184099960018, 33.018537000284745], [-96.64180999963804, 33.01860699963728], [-96.64178100022403, 33.018646999995624], [-96.64173500019326, 33.01867799997836], [-96.64167300026901, 33.018707000122475], [-96.64155299972009, 33.01874799997225], [-96.64150699968933, 33.0187800002092], [-96.64147400007737, 33.018819000199784], [-96.64139200006161, 33.019010999786055], [-96.64137199997012, 33.01904600003438], [-96.6413350001602, 33.01910099996606], [-96.64129900017518, 33.01912800015279], [-96.64125999981714, 33.019139999814634], [-96.64119100022016, 33.01913600017864], [-96.64112500009792, 33.01913199978916], [-96.6410680001965, 33.01913499989294], [-96.64095199984556, 33.019173999726554], [-96.64095100002064, 33.0192329999634], [-96.64074799963106, 33.019343000312375], [-96.64070699962316, 33.019500000209796], [-96.64074000013345, 33.01962699971596], [-96.64079000036216, 33.019815999721786], [-96.64079600020995, 33.01997200000716], [-96.64075299965391, 33.020134999677296], [-96.64059700001832, 33.020402999722904], [-96.64049300026123, 33.02048099975736], [-96.64039600017686, 33.02064899982891], [-96.64039000032906, 33.020735000279686], [-96.6403779997352, 33.02090700017632], [-96.6403019995672, 33.02121200004804], [-96.64024499966582, 33.0213199996657], [-96.6403559999939, 33.02150400002065], [-96.64061899975961, 33.02176399988388], [-96.64073699976039, 33.02205600028766], [-96.64055600001042, 33.02242800004081], [-96.64090199961683, 33.022691999666456], [-96.64090099979194, 33.02276999993525], [-96.64090099979194, 33.02283300024677], [-96.64090800036294, 33.02289299978426], [-96.64089999996702, 33.022951000302605], [-96.6409340003022, 33.02309400019695], [-96.6409979998763, 33.023200000042614], [-96.64106300017365, 33.023236999859265], [-96.64198000041567, 33.023962000013334], [-96.64208499999765, 33.02436500025062], [-96.64206900010413, 33.02449599967305], [-96.64174699988885, 33.024916999925246], [-96.64172699979736, 33.0250170000313], [-96.64183499975239, 33.02513999975766], [-96.64212000015775, 33.025257000274934], [-96.64212799965536, 33.026037999961765], [-96.64231900034937, 33.02630799979253], [-96.64238200009856, 33.02642900032419], [-96.64239399979414, 33.02653200007323], [-96.64238799994635, 33.02671499989349], [-96.64226699957253, 33.02706400034123], [-96.64217500040934, 33.027282999994206], [-96.64218399973184, 33.02738299967542], [-96.64215500031784, 33.02753299973762], [-96.64213799970109, 33.02756899980494], [-96.64202100042354, 33.02781100025242], [-96.64197500039279, 33.027997000231906], [-96.64197799986754, 33.02811200035483], [-96.64207300030208, 33.02828999975155], [-96.64219899980046, 33.02861299987964], [-96.64220100034859, 33.02873499968297], [-96.64215399959463, 33.028863999791376], [-96.64204800018773, 33.02906600031743], [-96.64175600010967, 33.02920500014951], [-96.64164200030687, 33.02929800022361], [-96.64146300020671, 33.029444000117614], [-96.64114400036449, 33.02968200002472], [-96.64089999996702, 33.02987000014886], [-96.640499000109, 33.03011899992419], [-96.64042200011612, 33.030204000013114], [-96.64041000042057, 33.030385000177354], [-96.64049999993394, 33.03048799982192], [-96.64060700006405, 33.03058499991816], [-96.64062700015553, 33.030754999868215], [-96.64061600028488, 33.030954000124005], [-96.64057000025413, 33.03111799997056], [-96.64048900006327, 33.03121500012647], [-96.6406580001177, 33.031256000183625], [-96.64049499991106, 33.032529000290594], [-96.64043399981169, 33.03339699982105], [-96.64042100029123, 33.03365300026744], [-96.64021500042689, 33.033709000242446], [-96.63985499967846, 33.033736999840045], [-96.63981000037094, 33.033906999734086], [-96.63973900022583, 33.03467799999645], [-96.63974599989852, 33.034722000274854], [-96.63977699986067, 33.03476200009805], [-96.63983700013512, 33.03480899990481], [-96.63993400021951, 33.03485400022506], [-96.64000299981647, 33.03490599976318], [-96.64006200026601, 33.03495900012984], [-96.64008400000732, 33.035010000253564], [-96.64008700038038, 33.03505100005109], [-96.6400749997865, 33.035082000289464], [-96.6400320001288, 33.03513800010931], [-96.63997500022738, 33.035200999864145], [-96.63979100010437, 33.03533000029732], [-96.63941500036071, 33.03553499985585], [-96.63932099975108, 33.03557899970629], [-96.63926200019985, 33.03561500025212], [-96.63916199974243, 33.035721000100544], [-96.63913599980316, 33.03576000012634], [-96.6390889999475, 33.035935000357895], [-96.63907800007684, 33.03601200010868], [-96.63907600042702, 33.03606599990349], [-96.63908700029768, 33.036112999767774], [-96.6391170004349, 33.03617199996242], [-96.63915800044278, 33.03623200020919], [-96.63921299979604, 33.03629199966201], [-96.63925399980393, 33.0363240003074], [-96.63942700005629, 33.03639399984304], [-96.63947399991196, 33.036433000323996], [-96.63951499991984, 33.03647500030131], [-96.63955900030078, 33.03652699963584], [-96.639589000438, 33.03658899981606], [-96.6395969999356, 33.03664199992337], [-96.63960000030866, 33.036732000174695], [-96.63961399965403, 33.0367920000401], [-96.63969900004285, 33.036918000151616], [-96.63976500016508, 33.03699900012818], [-96.63980299979993, 33.03703899967075], [-96.63984399980782, 33.037065999634116], [-96.639990000296, 33.03713500000498], [-96.6400749997865, 33.037166000263134], [-96.64016799967291, 33.03720799989103], [-96.64041600026835, 33.037166000263134], [-96.64063999967603, 33.03730600020005], [-96.64082299997412, 33.03737599970851], [-96.64087900005062, 33.03741300035143], [-96.64113199977061, 33.03764700027194], [-96.64132500011446, 33.037854999930474], [-96.64137299979504, 33.037874999886256], [-96.64142800004662, 33.03789800005571], [-96.64223399996057, 33.03823399975104], [-96.64238700012145, 33.038713999889914], [-96.64248099983277, 33.03900599986226], [-96.64278799997943, 33.03916200006976], [-96.64276799988795, 33.03920500031012], [-96.64225400005205, 33.0403150003361], [-96.64247699963481, 33.04202300012458], [-96.64310699982165, 33.04271800031783], [-96.6434759998926, 33.043132000020826], [-96.64356299993123, 33.04322099965374], [-96.64259800000863, 33.04328599967577], [-96.64251699981779, 33.044133000118585], [-96.64107900006715, 33.044047999912834], [-96.64096799973909, 33.04713100008464], [-96.64090099979194, 33.04900199996535], [-96.63939299972111, 33.048967000319564], [-96.6393500000634, 33.050370999873884], [-96.63931999992616, 33.05131699964707], [-96.63928399994117, 33.052492999794104], [-96.63977799968558, 33.0531329996974], [-96.64045100042844, 33.05299100026632], [-96.64059700001832, 33.05296000030703], [-96.64095600004352, 33.052934999749], [-96.6417360000182, 33.052879999851605], [-96.64312799973806, 33.0527820003053], [-96.64393399965199, 33.052724999750694], [-96.64400899999508, 33.052720000228625], [-96.64498700033647, 33.052865999708004], [-96.64553399978433, 33.05294700007798], [-96.64599800028985, 33.0530160000555], [-96.64611299991756, 33.05302699971956], [-96.64623799959104, 33.053031999977094], [-96.64734900000182, 33.053076000122964], [-96.64823699993153, 33.05311099964148], [-96.65009599963163, 33.05318399984018], [-96.65015300043135, 33.05318700029059], [-96.6505860000764, 33.053203000181114], [-96.65099699998014, 33.05290900003455], [-96.65205999981202, 33.0521549997045], [-96.65240200011878, 33.05190900020911], [-96.6528740001219, 33.05157499980646], [-96.65308400018418, 33.051462999946835], [-96.6531489995832, 33.0513190002407], [-96.65380300005948, 33.050744999668105], [-96.65396100024324, 33.0508139998653], [-96.65403599968798, 33.05086000021766], [-96.65411199985597, 33.05091799964561], [-96.65422900003183, 33.051014999836845], [-96.65433000031418, 33.051113999766564], [-96.65440800013198, 33.051180999830784], [-96.65446199966034, 33.051255999965875], [-96.65451600008703, 33.051357000217415], [-96.65459100043009, 33.05146499978425], [-96.65472699997422, 33.05159600032898], [-96.65479300009646, 33.05167399985954], [-96.65487099991425, 33.05175000024306], [-96.65491000027232, 33.05179700002141], [-96.65493399966344, 33.05185299975659], [-96.6549579999529, 33.051963000047046], [-96.65497600039456, 33.0520220001667], [-96.65500900000653, 33.052075000023606], [-96.65506300043319, 33.052132000246026], [-96.65514200007591, 33.052193999983785], [-96.65522900011455, 33.052244999917505], [-96.65557399989605, 33.05241299966734], [-96.65562199957661, 33.05245000001735], [-96.6558030002249, 33.05262899984279], [-96.6559229998755, 33.05273999982098], [-96.65598599962469, 33.052805000346936], [-96.65604900027219, 33.052902999867655], [-96.65607900040943, 33.052958999652375], [-96.65618499981632, 33.05304999969721], [-96.65620900010578, 33.053075000222584], [-96.65626600000718, 33.05318100014263], [-96.65628100007578, 33.0532209998662], [-96.65634100035024, 33.05330199990961], [-96.65635600041887, 33.053361999782176], [-96.65635600041887, 33.05341600023487], [-96.65636200026664, 33.05345499995561], [-96.65639200040388, 33.05350900035126], [-96.65639400005371, 33.053569000082774], [-96.65638900003083, 33.05361999997301], [-96.65639800025166, 33.05366900004639], [-96.65644000008444, 33.053859999857], [-96.65648899958993, 33.053925000309086], [-96.65654300001661, 33.05397300032248], [-96.65659800026819, 33.05399899972279], [-96.65668699995665, 33.054039999717126], [-96.65673600036047, 33.05407100004913], [-96.65678099966799, 33.05411100012126], [-96.65685299963802, 33.05423999996674], [-96.65689000034624, 33.05437200003318], [-96.65689700001894, 33.05447600012865], [-96.65694100039987, 33.05472000021804], [-96.65699599975315, 33.05484999980434], [-96.65706799972315, 33.055002000293534], [-96.6571060002563, 33.05506400026911], [-96.65712299997475, 33.05513100033935], [-96.65711999960169, 33.05522400015011], [-96.65710899973105, 33.055307000359704], [-96.6571050004314, 33.055415000351985], [-96.65710299988328, 33.05550100000008], [-96.65709200001261, 33.05557299985104], [-96.65703599993613, 33.05566599994778], [-96.65698900008044, 33.05571999974029], [-96.65684799961514, 33.05583300007733], [-96.65673800001028, 33.05591100037191], [-96.65664099992591, 33.05597299970736], [-96.65657300015384, 33.05601900037619], [-96.65641800034314, 33.05614000000749], [-96.6562330003952, 33.05630000019175], [-96.65586400032426, 33.056603999665434], [-96.65574200012553, 33.056715999761664], [-96.65542499993312, 33.057048999629785], [-96.65521700041899, 33.05728300008589], [-96.65509000019738, 33.05747100003389], [-96.65506699973284, 33.057528999624395], [-96.6550490001895, 33.05759199992822], [-96.65504400016663, 33.05770400027333], [-96.65506099988505, 33.057774999981326], [-96.65517100038822, 33.05794600000809], [-96.65519999980224, 33.05800300019415], [-96.65521099967287, 33.05806700000246], [-96.65525699970362, 33.05824099974013], [-96.65530899958218, 33.05847800030945], [-96.65530899958218, 33.058517000046564], [-96.65528600001596, 33.05858799985175], [-96.65528000016818, 33.05864700004926], [-96.65528600001596, 33.05872000017515], [-96.65532100017604, 33.058880000187735], [-96.65534399974227, 33.05895700020773], [-96.65536100035902, 33.05903499999314], [-96.6554020003669, 33.059132000276556], [-96.65544800039766, 33.059213000151935], [-96.65550600012398, 33.05927199993055], [-96.65559800018549, 33.05933399991053], [-96.65567900037634, 33.05943499993014], [-96.65572500040709, 33.059503000265664], [-96.65573700010268, 33.05956199984997], [-96.65575999966889, 33.0596099998016], [-96.65580599969964, 33.05964499973415], [-96.65586400032426, 33.05966900006701], [-96.65589299973827, 33.05968799976123], [-96.65591000035502, 33.05971100025737], [-96.65592700007346, 33.059780000204], [-96.65596200023356, 33.05985200031973], [-96.65600800026432, 33.05990499998041], [-96.6560280003558, 33.059960999828725], [-96.65612299989205, 33.06022499992429], [-96.65615200020436, 33.06033600011305], [-96.6561689999228, 33.060365999916215], [-96.65622099980132, 33.06041400018226], [-96.65626699983208, 33.06043699973581], [-96.6562960001444, 33.060476999927104], [-96.65631899971062, 33.060521000116545], [-96.6563379999772, 33.06060600023198], [-96.65635499969562, 33.060676000044], [-96.65636599956629, 33.06084999989475], [-96.65633600032737, 33.06147899991747], [-96.65631299986283, 33.06155199995337], [-96.65628400044885, 33.06160000032556], [-96.65620699955764, 33.06165100008069], [-96.65609399957971, 33.061726000331106], [-96.65606000014286, 33.061784000129634], [-96.65603699967832, 33.06189099990425], [-96.65603699967832, 33.062181000229614], [-96.6560140001121, 33.06222900025883], [-96.65588699989048, 33.06229200021033], [-96.65584099985973, 33.06235999984451], [-96.65582300031639, 33.0624519997531], [-96.65581200044574, 33.06255799967586], [-96.65579999985187, 33.06262599985741], [-96.6557889999812, 33.062733000114186], [-96.65579999985187, 33.062804000283116], [-96.65582900016418, 33.06286299991339], [-96.65594999963967, 33.062994000061195], [-96.65604900027219, 33.063046000301426], [-96.65614600035659, 33.0630799998338], [-96.65627900042597, 33.0631329998094], [-96.65639400005371, 33.06318700029152], [-96.65649899963567, 33.063273999660154], [-96.6565570002603, 33.063360999695604], [-96.65659700044328, 33.063442999979195], [-96.65677099962225, 33.064062000278845], [-96.656788000239, 33.064140000057584], [-96.65684000011753, 33.06426600019119], [-96.6568800003005, 33.06438699977931], [-96.65695499974525, 33.06451199978882], [-96.65704799963167, 33.06460999982875], [-96.65711700012697, 33.064691999701445], [-96.65716300015772, 33.06475999973991], [-96.65718700044717, 33.06481799979763], [-96.65719199957171, 33.064900000229386], [-96.65718099970105, 33.06506400011084], [-96.6571349996703, 33.065320000361574], [-96.65709399966244, 33.06544100000625], [-96.6570419997839, 33.06564900026969], [-96.65703599993613, 33.065767000193325], [-96.65704799963167, 33.06592499993263], [-96.65704500015694, 33.066077000119336], [-96.65705699985251, 33.06619299996635], [-96.65704000013407, 33.06640099995808], [-96.65698900008044, 33.06651299963419], [-96.6569089997145, 33.06659799998556], [-96.65681400017826, 33.06668499974651], [-96.65653100032104, 33.06691999992553], [-96.65610500034869, 33.06730099985936], [-96.6555499996066, 33.067937000081], [-96.655325000374, 33.068196000194106], [-96.65524199963504, 33.068340000148126], [-96.65514600027387, 33.06860999980325], [-96.65511699996155, 33.06869499963476], [-96.65510799974072, 33.06880499995905], [-96.65509400039534, 33.06900999988853], [-96.65510000024311, 33.06929099963647], [-96.65511699996155, 33.069629000177834], [-96.6550709999308, 33.0697410002624], [-96.65497300002151, 33.06988100035577], [-96.6549379998614, 33.06997799997799], [-96.6549379998614, 33.07033499979307], [-96.65507199975572, 33.07051299968263], [-96.65511100011378, 33.07061200026896], [-96.65519300012954, 33.07081500035481], [-96.65523499996233, 33.07100000016238], [-96.65520900002305, 33.07118000035455], [-96.65520700037324, 33.07119500029123], [-96.65518899993158, 33.07128900013724], [-96.65516499964212, 33.07141499982129], [-96.65511500031174, 33.071526999890864], [-96.65504400016663, 33.07168799964699], [-96.65497999969419, 33.071795000177566], [-96.65486099986852, 33.07202300020581], [-96.65481800021081, 33.07215299983197], [-96.6547899997234, 33.072356000126405], [-96.6548319995562, 33.07259199989476], [-96.6548459997999, 33.07266400006807], [-96.65486000004361, 33.07270499964907], [-96.6549029997013, 33.0727949999291], [-96.65496999964846, 33.07286099962429], [-96.65510000024311, 33.072991000270555], [-96.65524699965789, 33.073129000364275], [-96.6555570001776, 33.073035000225765], [-96.65582800033926, 33.07295400027408], [-96.65618700036445, 33.07284300019132], [-96.65648799976503, 33.07275000017698], [-96.65662699968219, 33.072717000293935], [-96.65679099971372, 33.07267800007405], [-96.65693399982885, 33.07265400027753], [-96.65703799958594, 33.07264300005495], [-96.6572459999984, 33.072633999940315], [-96.65730899974757, 33.07263300026204], [-96.65833899996754, 33.07264099994593], [-96.65861199977898, 33.07264899962905], [-96.6591039998736, 33.072675000288015], [-96.65937000001236, 33.07272299986298], [-96.65988700022132, 33.07287399992337], [-96.66013400009352, 33.07287399992337], [-96.66321499963416, 33.0728539996361], [-96.66345700038181, 33.072848999751834], [-96.6634699999023, 33.07284500029581], [-96.66359200010103, 33.07281899968709], [-96.66395100012625, 33.072736000182466], [-96.66416100018853, 33.072696000293476], [-96.66441800010647, 33.07267100007131], [-96.66529299961738, 33.072650000059866], [-96.66547699974039, 33.07261700013933], [-96.66575299992493, 33.072601999692345], [-96.66633999955576, 33.07256699964309], [-96.66682400015273, 33.072546000359615], [-96.66742400020236, 33.07256599996405], [-96.66824400036002, 33.07259199989476], [-96.66867500035524, 33.072605999912184], [-96.66928499955227, 33.07261700013933], [-96.66997400018869, 33.07261999992734], [-96.67128800009075, 33.072638999836855], [-96.67285899991076, 33.07266199995952], [-96.67425999985146, 33.07270800018681], [-96.67509699972754, 33.07274699964062], [-96.67575000037894, 33.07279200014705], [-96.67603200041124, 33.07281500022972], [-96.67667699976839, 33.07287399992337], [-96.677064000281, 33.072917999937076], [-96.68079299955184, 33.07306799998952], [-96.68182899961957, 33.07310499993806], [-96.68218000014716, 33.07312099997197], [-96.68234099980565, 33.073001999696835], [-96.68330599972825, 33.07174000001958], [-96.68347300013282, 33.07151999979661], [-96.68384899987646, 33.07103599977862], [-96.68395900037963, 33.070896999715295], [-96.68400399968716, 33.07089800016604], [-96.68407999985514, 33.070880000331506], [-96.68422999964297, 33.07083100005241], [-96.68438100015402, 33.07081100005354], [-96.68446599964453, 33.070813999903116], [-96.6845689995767, 33.07081100005354], [-96.68469900017135, 33.070818999903096], [-96.68479600025572, 33.070813999903116], [-96.68492799960188, 33.07084100005014], [-96.68506099967126, 33.07087300018577], [-96.68514900043313, 33.07091699969456], [-96.6852380001216, 33.070990999879605], [-96.68528499997726, 33.07105799987142], [-96.68531699976431, 33.071126000260286], [-96.68534699990154, 33.07115899998725], [-96.68534400042681, 33.07120099996406], [-96.68531099991652, 33.07124700020244], [-96.68526999990866, 33.071272000076384], [-96.68521100035743, 33.07128900013724], [-96.68509900020442, 33.071350999932605], [-96.68496100011215, 33.07144699974817], [-96.68473699980618, 33.07156099991064], [-96.68461299995762, 33.07164000026017], [-96.6845340003149, 33.071723999734466], [-96.68449999997972, 33.07179000023305], [-96.68442800000969, 33.071845000359794], [-96.68434099997104, 33.07199000005009], [-96.68430499998604, 33.072059999841024], [-96.68430499998604, 33.07210299982478], [-96.68432000005465, 33.0721330001336], [-96.68437699995606, 33.072197999889426], [-96.68442800000969, 33.072280000219344], [-96.68443300003256, 33.07234699998232], [-96.68442299998682, 33.07240300019457], [-96.68438199997894, 33.072497999935294], [-96.68428499989456, 33.07259499968362], [-96.68427900004679, 33.07264300005495], [-96.68431000000892, 33.0727039999716], [-96.68433599994819, 33.07273899996642], [-96.6843870000018, 33.07275999995655], [-96.68443300003256, 33.07278900036496], [-96.68445399994896, 33.07283699987768], [-96.68447200039063, 33.07289799965982], [-96.68447999988823, 33.072922999817486], [-96.68447999988823, 33.07304000009952], [-96.68452599991899, 33.073267000241536], [-96.6845689995767, 33.0733599997094], [-96.68458199999547, 33.07341699969897], [-96.68459200004122, 33.07356800007377], [-96.68460299991187, 33.07366300006212], [-96.68462300000336, 33.073726000108756], [-96.68467999990477, 33.07377899969488], [-96.68473599998126, 33.07379100019326], [-96.68481299997416, 33.073787999692456], [-96.68483300006565, 33.07378499994427], [-96.68487499989844, 33.07377899969488], [-96.68495199989133, 33.073739999945744], [-96.68499299989921, 33.07373500011175], [-96.6850489999757, 33.07373500011175], [-96.6851259999686, 33.07375300011494], [-96.68518299987001, 33.073739999945744], [-96.68522399987789, 33.07371500002027], [-96.68527499993151, 33.07370600001522], [-96.68532099996226, 33.07370600001522], [-96.68536199997016, 33.073726000108756], [-96.68540299997804, 33.07375700028245], [-96.6855059999102, 33.07379100019326], [-96.68555500031401, 33.07386399980239], [-96.68555500031401, 33.07392600004156], [-96.68553000019965, 33.07397700018125], [-96.68554500026825, 33.07403399977103], [-96.68559999962152, 33.07408099969554], [-96.6856890002083, 33.07413700030957], [-96.68570400027693, 33.074173999808544], [-96.6856540000482, 33.07450899993481], [-96.68566400009395, 33.07455499994923], [-96.68570900029978, 33.07460500006843], [-96.68592199983681, 33.07479900008143], [-96.68593199988254, 33.07484499994415], [-96.68591699981393, 33.07488199989809], [-96.68589199969958, 33.07491200001213], [-96.68580300001112, 33.07498700025237], [-96.68567400013968, 33.075114999962686], [-96.6856540000482, 33.075162000062434], [-96.68566400009395, 33.07519099967902], [-96.68570900029978, 33.07521899963722], [-96.6857529997824, 33.07524100018953], [-96.68603499981471, 33.07530199981039], [-96.68622899998346, 33.07536399978893], [-96.6863079996262, 33.075194999781154], [-96.68706500003464, 33.07335200009093], [-96.68753500038792, 33.07217899988473], [-96.68811000032319, 33.07075000033009], [-96.68825199971509, 33.07042099992], [-96.68927799973707, 33.06784700010157], [-96.68931500044532, 33.067754999809985], [-96.69002099990183, 33.066077000119336], [-96.69007699997832, 33.065954000335225], [-96.6901520003214, 33.06577300023625], [-96.6903280000485, 33.06534000010278], [-96.69077800031027, 33.06422200036819], [-96.69239699981088, 33.06020200031535], [-96.69265399972882, 33.059590000264095], [-96.6927859999733, 33.05927199993055], [-96.69295500002771, 33.05892100016764], [-96.69324799993068, 33.058310000086095], [-96.6936510003368, 33.05747699988943], [-96.69370999988803, 33.05737000001959], [-96.69380799979731, 33.057222999873794], [-96.69387599956939, 33.05710100037048], [-96.69443700015925, 33.056039999838774], [-96.69473799955983, 33.05550000012724], [-96.69485300008587, 33.05528699984463], [-96.69561200014415, 33.05386599995864], [-96.69589500000137, 33.053305000355984], [-96.69600200013147, 33.0531060001414], [-96.69622899991221, 33.05268800026914], [-96.69869400040803, 33.04802899981108], [-96.69943000000178, 33.046667000203435], [-96.69951500039059, 33.046491000232145], [-96.70197400014032, 33.041903999975055], [-96.70219300042343, 33.04149699999498], [-96.7024720000827, 33.040969999698795], [-96.70254900007559, 33.040837999875286], [-96.70305100021591, 33.03987499964486], [-96.70343299980733, 33.039129999702176], [-96.70426300001073, 33.037580999778], [-96.7043309997828, 33.03745900017248], [-96.70453300034745, 33.03705500023578], [-96.70502399971882, 33.03612999984066], [-96.70561799992066, 33.034992999964565], [-96.70577899957915, 33.034710999830786], [-96.70591800039463, 33.034456000310705], [-96.70633199977311, 33.03377999971574], [-96.70646700039065, 33.03352300026901], [-96.70683399991346, 33.03275299982189], [-96.70732199981006, 33.03181499986733], [-96.70771599999537, 33.031059000157356], [-96.70815600021143, 33.03023700001669], [-96.70821300011283, 33.030140000290494], [-96.70850699984071, 33.029525000186595], [-96.70866500002445, 33.02916399989129], [-96.70895200007963, 33.02856000021494], [-96.70897700019398, 33.02848699994251], [-96.70899599956225, 33.0284189997581], [-96.70915799994395, 33.02794900037198], [-96.70927699976964, 33.02757100018485], [-96.70934999956457, 33.02731500014556], [-96.70955899980196, 33.02647200000009], [-96.70975100032088, 33.02543599962635], [-96.70982499994072, 33.02474599978979]]]}, "properties": {"ZCTA5": "75074", "AREALAND": 41384910, "AREAWATER": 133528}}, {"type": "Feature", "geometry": {"type": "Polygon", "coordinates": [[[-96.86316300012703, 33.05862899996767], [-96.86297999982895, 33.05859000028019], [-96.86232600025095, 33.05836899971706], [-96.86153899970527, 33.05822500001225], [-96.85984700040976, 33.058052000086825], [-96.85915199992557, 33.05798300029155], [-96.85865900000606, 33.05791299984385], [-96.85838899966933, 33.057883999803444], [-96.8580830002459, 33.0578639998738], [-96.85786099958973, 33.05784399993963], [-96.85780399968833, 33.05821299964981], [-96.85769099971041, 33.05907699973315], [-96.85758199993046, 33.05944899978207], [-96.8575350000748, 33.05973099976738], [-96.8575350000748, 33.05987699966648], [-96.85745899990681, 33.06053899981125], [-96.85742999959452, 33.060712999932875], [-96.85741299987608, 33.06081300006346], [-96.8571690003769, 33.060818999691215], [-96.85718299972231, 33.060712999932875], [-96.85688700034461, 33.060716000126625], [-96.85646899986986, 33.06074400017374], [-96.85556599987153, 33.060765999828035], [-96.85518500010504, 33.060770999646024], [-96.85413899999156, 33.060770999646024], [-96.85353499974399, 33.060770999646024], [-96.85343900038282, 33.06073799978792], [-96.85341000007053, 33.06068999969855], [-96.85340000002478, 33.06064200033585], [-96.85338999997902, 33.060253000127524], [-96.85337599973532, 33.060195000073065], [-96.85325000023694, 33.060187000010615], [-96.85161499994452, 33.060169000243924], [-96.85118899997217, 33.06018399979884], [-96.85093299987912, 33.060186000191], [-96.84953100011353, 33.060169000243924], [-96.84880600039043, 33.060154999753614], [-96.8481140002793, 33.06014199983411], [-96.8470089997163, 33.0601339997669], [-96.84695099998996, 33.0601339997669], [-96.84679000033148, 33.060132999946624], [-96.84663399979756, 33.06014100001397], [-96.84617699986308, 33.06014500004738], [-96.84521000029063, 33.060110000313635], [-96.84370600041778, 33.060121999664645], [-96.84293599959054, 33.060099999849385], [-96.84280300041947, 33.06013200012638], [-96.84236700040137, 33.060139000373674], [-96.8424259999526, 33.05875999964087], [-96.84244900041713, 33.05822800029087], [-96.8425039997704, 33.05695000034996], [-96.84241599990685, 33.05693799981372], [-96.84220500001965, 33.056910000061826], [-96.84199800033042, 33.05688300015824], [-96.84129600017356, 33.056848999738], [-96.84121900018064, 33.056846000165315], [-96.8410250000119, 33.05686200014381], [-96.84095499969169, 33.05686899989817], [-96.8403279998779, 33.056873000080245], [-96.83880599956336, 33.056854999635924], [-96.83856299988912, 33.05684799988047], [-96.83846800035289, 33.0568429998396], [-96.83837399974323, 33.056841000124386], [-96.8383789997661, 33.056979999801285], [-96.83835799984972, 33.05790200003736], [-96.83837000044359, 33.05802999975425], [-96.8384130001013, 33.05830000017026], [-96.83843999986549, 33.058434999691244], [-96.8384470004365, 33.05849999977717], [-96.83853299975189, 33.05887500026233], [-96.8385479998205, 33.05905299999171], [-96.83854199997273, 33.05921499981327], [-96.83286299986686, 33.05917999970976], [-96.83287200008769, 33.05902599999247], [-96.83288100030853, 33.05888199985663], [-96.83290600042288, 33.05843999964164], [-96.83289499965392, 33.05717599998025], [-96.83290600042288, 33.05665799963553], [-96.83012100025992, 33.05668500036104], [-96.83009799979538, 33.05667699997962], [-96.83006199981037, 33.05683100004169], [-96.8299930002134, 33.05709100031732], [-96.82991899969524, 33.05732900007017], [-96.8297959996716, 33.057672999714796], [-96.82973100027259, 33.05784099964799], [-96.82964400023393, 33.0580399997008], [-96.82945900028602, 33.05842999974057], [-96.82926699976707, 33.0587710000931], [-96.82916399983492, 33.058945999778786], [-96.829014999872, 33.0591749998014], [-96.82887000010705, 33.05937600026077], [-96.82865300037207, 33.05966800024151], [-96.82844300030979, 33.059955000142416], [-96.82823300024751, 33.06024000022248], [-96.82804599975144, 33.060484999963194], [-96.82751600002202, 33.0612069998488], [-96.8270640001104, 33.06183000002055], [-96.82658999955916, 33.062461000156354], [-96.8261989997469, 33.06300999967507], [-96.82588699957736, 33.06347600032875], [-96.82565800014683, 33.06385899967151], [-96.82550400016103, 33.064174999694536], [-96.82544999973436, 33.06431799967996], [-96.82534599997729, 33.0645739998421], [-96.82521799993077, 33.06493699984834], [-96.82511900019657, 33.065308999975], [-96.82508499986137, 33.06545999996023], [-96.82504999970128, 33.06561500022405], [-96.82477500023998, 33.06559600030353], [-96.82471400014062, 33.06559600030353], [-96.82464200017058, 33.06559600030353], [-96.82456600000262, 33.06559799981984], [-96.82446300007045, 33.0656020003581], [-96.82423700011464, 33.0656030001162], [-96.82409700037256, 33.065603999874284], [-96.82349200030006, 33.0656030001162], [-96.82297500009112, 33.065593000276046], [-96.82134300017174, 33.0656020003581], [-96.81876999972087, 33.06559400003425], [-96.81876999972087, 33.065419999774996], [-96.81876100039838, 33.065084999674184], [-96.81876100039838, 33.06473600004232], [-96.81881099972877, 33.06446199995691], [-96.81890199996536, 33.064230000063894], [-96.81902699963884, 33.063996999783], [-96.8192009997161, 33.063780999643676], [-96.8194250000221, 33.06354099970366], [-96.81962400021372, 33.06337499967598], [-96.81995700029965, 33.06320900008807], [-96.82032200017262, 33.06306800013317], [-96.82060400020494, 33.062968000305965], [-96.82102699980423, 33.06289300036097], [-96.8213759997837, 33.06283499978755], [-96.82171600044063, 33.06275199989991], [-96.82199000007702, 33.062668999934004], [-96.82228899982776, 33.06254500011124], [-96.82254699957062, 33.06237000005199], [-96.82282899960293, 33.0621630001178], [-96.82305299990892, 33.06189700021142], [-96.82324399970464, 33.06163100025403], [-96.82340199988838, 33.06129900020966], [-96.82346800001062, 33.06102499996459], [-96.82350099962258, 33.06081799987878], [-96.82353199958472, 33.060586999983016], [-96.82353300030795, 33.06018500037136], [-96.82353300030795, 33.05987099997451], [-96.82353900015572, 33.059587999858394], [-96.82353900015572, 33.05943099986447], [-96.82357000011785, 33.059249000072604], [-96.82367700024798, 33.05900399990331], [-96.823777999632, 33.05879099981668], [-96.82394699968643, 33.05855199966077], [-96.82419799975658, 33.05826300001998], [-96.82494000009642, 33.057420999966745], [-96.82532300041107, 33.05699400004365], [-96.82548599971936, 33.0568179996313], [-96.82560000042051, 33.05666799973788], [-96.8292890004067, 33.05668999965791], [-96.82941400008018, 33.05590499965671], [-96.82944000001945, 33.05539100036514], [-96.82944000001945, 33.05524999993389], [-96.82944699969212, 33.05484800004386], [-96.82869600002977, 33.054842000009096], [-96.82842700041626, 33.054790000192895], [-96.82807599988867, 33.054691999760735], [-96.82628300031081, 33.05465800000021], [-96.82604499976112, 33.05465399971749], [-96.82461200003337, 33.05469600004173], [-96.82381799981498, 33.05468399995114], [-96.82349999979768, 33.054747000032044], [-96.82308200022123, 33.05486500026554], [-96.82272300019604, 33.05502000036378], [-96.82228800000284, 33.05531500011271], [-96.8218200001977, 33.05572100036356], [-96.82149399978447, 33.05601500015531], [-96.82120599990438, 33.05617900002743], [-96.82097300027588, 33.05630299978305], [-96.82073899992415, 33.05640400026855], [-96.820375999701, 33.05651900020314], [-96.82010799991238, 33.05657099975106], [-96.8198690004361, 33.05658900025344], [-96.81907799969245, 33.05655800005518], [-96.81602999976376, 33.05655800005518], [-96.81303100023888, 33.056565999694506], [-96.81288499975068, 33.05656499983377], [-96.81268399990924, 33.05657099975106], [-96.81263799987848, 33.056572000364646], [-96.81182600011675, 33.056559999776894], [-96.81111300008922, 33.056572000364646], [-96.81058199963658, 33.056559999776894], [-96.80963799963037, 33.056565999694506], [-96.80918800026689, 33.05664699967217], [-96.80865699981423, 33.056841000124386], [-96.80819499985687, 33.05712800020722], [-96.80778099958009, 33.05750299976076], [-96.80760400002808, 33.05764900034349], [-96.80742800030099, 33.05755699993232], [-96.80712000032942, 33.05741099994992], [-96.80684900016776, 33.057289999806756], [-96.80660499977029, 33.057197999773344], [-96.80625300031609, 33.05709300002685], [-96.80613500031531, 33.05705199994842], [-96.80593000027591, 33.05699499989952], [-96.80568599987843, 33.056936000100606], [-96.80549900028069, 33.056894000092036], [-96.80524700038562, 33.056846000165315], [-96.80465400000868, 33.056774999708026], [-96.80435199988489, 33.05675500027941], [-96.80396199989755, 33.056740999998986], [-96.80350800033611, 33.0567450001869], [-96.8034160002746, 33.0567419998578], [-96.80247399991822, 33.05669700017892], [-96.80067299994444, 33.05669399984813], [-96.80052499980643, 33.05669299998883], [-96.79839199957163, 33.056673999648204], [-96.79569199979748, 33.05665199972412], [-96.79537200013034, 33.05663200026758], [-96.79518100033464, 33.05662600035438], [-96.79513500030387, 33.05646600000967], [-96.79145100034056, 33.05640099992777], [-96.79114900021676, 33.05639199965781], [-96.7906799996884, 33.056387000344024], [-96.79019799963955, 33.05638999993231], [-96.78940699979421, 33.056376000346816], [-96.78929800001426, 33.05637100027919], [-96.78910700021855, 33.056362000006175], [-96.78771200012564, 33.056335999797874], [-96.78732399978813, 33.05633099972793], [-96.78643099983555, 33.05631700013306], [-96.78603099980245, 33.05632200020375], [-96.78542999992791, 33.0563120000621], [-96.78366300028934, 33.05629199977533], [-96.78304999982092, 33.05628199963028], [-96.78151199961286, 33.05626100022542], [-96.7811199999757, 33.05626600029929], [-96.780408999598, 33.05625199994116], [-96.78029999981806, 33.05625000021245], [-96.77894200043337, 33.05622999991161], [-96.7779230000841, 33.05621600030068], [-96.77697700042806, 33.05620899974142], [-96.77677900006134, 33.056206000146986], [-96.77656599962603, 33.05620399966433], [-96.77633800002037, 33.05619799972203], [-96.77596999977436, 33.056190000049256], [-96.77453600022169, 33.05618200037577], [-96.77395100024067, 33.05617599967905], [-96.7737269999347, 33.05616000032879], [-96.77170199965494, 33.055940000309384], [-96.7715100000343, 33.055936999952856], [-96.77136500026936, 33.05594299991295], [-96.77094799961951, 33.05598100015277], [-96.77052599984512, 33.05600900020013], [-96.77044800002733, 33.05601000006719], [-96.77037100003442, 33.05601199980133], [-96.77039500032387, 33.05673099990494], [-96.7704110002174, 33.057774999981326], [-96.7704050003696, 33.05921999971936], [-96.77039899962352, 33.06058600016794], [-96.7703969999737, 33.061053999723114], [-96.77038799975286, 33.06159399999815], [-96.77039199995082, 33.061924999882834], [-96.77039399960064, 33.06245099995921], [-96.77039899962352, 33.06347000012924], [-96.77039500032387, 33.064097999722], [-96.77039000030099, 33.064332000260904], [-96.7703769998822, 33.06529099975007], [-96.77037500023238, 33.065798000159766], [-96.77035900033886, 33.06733799997449], [-96.7703559999658, 33.06748300031651], [-96.77036000016378, 33.06768399987107], [-96.77038699992795, 33.068060999835], [-96.7704050003696, 33.06818500016702], [-96.77043299995871, 33.06829500037605], [-96.77049700043112, 33.068511999700775], [-96.77053300041614, 33.06860999980325], [-96.77057900044689, 33.06871700031286], [-96.77071699964084, 33.06899999968272], [-96.7710389998561, 33.06948500007479], [-96.77108599971179, 33.069553000173734], [-96.77145700033086, 33.07009399995234], [-96.77170599985288, 33.070466000110464], [-96.77177600017309, 33.0705920002203], [-96.77185399999088, 33.07077800019605], [-96.77197600018962, 33.07116600011028], [-96.77201100034974, 33.0713300003589], [-96.77202899989308, 33.07146199963933], [-96.77203899993881, 33.071725999864405], [-96.77202899989308, 33.07189299963478], [-96.77201600037259, 33.07373500011175], [-96.77199299990806, 33.07453799976671], [-96.77193799965647, 33.075764000032606], [-96.77188199957999, 33.07681499983582], [-96.77184899996801, 33.07767700021767], [-96.77180999960996, 33.07859800005117], [-96.77181100033319, 33.078663000299514], [-96.7717900004168, 33.07883399963497], [-96.77176500030244, 33.07893899978844], [-96.77174699986077, 33.079070000090994], [-96.77172400029454, 33.07929600012559], [-96.7717069996778, 33.07980000008306], [-96.77168699958631, 33.0804120003426], [-96.771672000416, 33.082646999893896], [-96.77166900004296, 33.082906999920695], [-96.77163999973064, 33.08458100034791], [-96.77162800003507, 33.08594899980345], [-96.77162800003507, 33.08610599990204], [-96.77163600043099, 33.08748799962598], [-96.77163100040814, 33.08825399983958], [-96.77162399983712, 33.08941200034714], [-96.7715800003545, 33.09125900007746], [-96.7715579997149, 33.092788999973855], [-96.77154400036949, 33.093306999854455], [-96.77154600001931, 33.09387399995736], [-96.77152099990496, 33.09531200018306], [-96.77152000008005, 33.0957050003367], [-96.77151900025513, 33.09620999973363], [-96.77152299955478, 33.09630200036794], [-96.77152699975274, 33.096910000306686], [-96.77152099990496, 33.09783500022765], [-96.77150699966126, 33.099102000147695], [-96.77150400018651, 33.099411999995624], [-96.7714949999657, 33.09988200020418], [-96.77149900016366, 33.10052699991372], [-96.77148200044522, 33.101004000185846], [-96.77148099972199, 33.101077999890435], [-96.77148699956976, 33.101166000271526], [-96.7714749998742, 33.10124599972059], [-96.7714749998742, 33.101373000239064], [-96.77145499978273, 33.10244299975019], [-96.7714530001329, 33.10256599968195], [-96.77142499964549, 33.103194000101055], [-96.77140000042945, 33.1033919998299], [-96.771368999569, 33.10354099974974], [-96.77134900037582, 33.10359799974322], [-96.77128800027646, 33.10377199973214], [-96.77123899987266, 33.10388700021035], [-96.77118600016921, 33.10397299988724], [-96.77107700038927, 33.10416799990169], [-96.77095199981747, 33.10434300033727], [-96.77081899974809, 33.10451099994603], [-96.77070999996815, 33.10462700027673], [-96.77049499988297, 33.10483300030495], [-96.77035799961563, 33.10494499995692], [-96.77021799987355, 33.10504700034889], [-96.76996000013068, 33.105219999665955], [-96.76942299983025, 33.105563000118984], [-96.76921899961575, 33.10568900036383], [-96.76902600017021, 33.105861999921906], [-96.76885899976564, 33.10600299975779], [-96.76871999984847, 33.10613699977107], [-96.76861899956613, 33.10627799991817], [-96.76843600016635, 33.106622000048226], [-96.76834999995262, 33.10675499982715], [-96.76827600033278, 33.106893999652584], [-96.7682429998225, 33.10696999974566], [-96.76818500009618, 33.107121999734616], [-96.76818300044634, 33.107158999652505], [-96.7681790002484, 33.10723300019403], [-96.76813800024051, 33.10804100036391], [-96.76807700014115, 33.10921800009591], [-96.76805700004965, 33.1101890000818], [-96.76803000028548, 33.1115360000857], [-96.76802400043769, 33.112081000364576], [-96.76802000023974, 33.112402000091166], [-96.76812099962376, 33.11235900024979], [-96.76822799975389, 33.1123190003259], [-96.76838699976254, 33.11226000000987], [-96.76894300032954, 33.1120520001229], [-96.77114000013846, 33.111204000290996], [-96.77178399967069, 33.1109309997374], [-96.77199499955789, 33.1108430001058], [-96.7738829995703, 33.11006800029616], [-96.78399600037525, 33.1060240000822], [-96.7841679999044, 33.10595500028735], [-96.78803299961768, 33.104313999798975], [-96.78808200002149, 33.10430099963239], [-96.78816099966421, 33.10426999969052], [-96.7882650003196, 33.10423600027576], [-96.78837499992446, 33.104191000056204], [-96.78854499980376, 33.104123999720144], [-96.79099199985794, 33.10315100026822], [-96.7928480000833, 33.102412999997625], [-96.7941929999475, 33.1018630002114], [-96.79542599965869, 33.101348999995146], [-96.79583699956244, 33.10118800035304], [-96.79678899996456, 33.10081599994441], [-96.79749199994635, 33.10055299985642], [-96.79755999971842, 33.10053000025481], [-96.79763300041166, 33.10049600014561], [-96.79781699963637, 33.10041000007634], [-96.80034700042951, 33.0993670003161], [-96.80341000042681, 33.098083999695575], [-96.80403800006555, 33.0978009998285], [-96.804563999597, 33.09757799962911], [-96.8049650003533, 33.09742000029958], [-96.8051069997452, 33.09736300006209], [-96.80516700001968, 33.09734099977571], [-96.80538899977752, 33.097254999629634], [-96.80553899956533, 33.09711100035173], [-96.80592000023017, 33.096957000047645], [-96.80605199957633, 33.09690299997487], [-96.80638200018754, 33.09676800002403], [-96.80723199958409, 33.09642200013442], [-96.80859499988996, 33.09593099976528], [-96.80870899969277, 33.09579900032941], [-96.81026100014454, 33.09483399969828], [-96.81207499963878, 33.094023999643646], [-96.81305600035324, 33.093702000323155], [-96.81429099971426, 33.093297000152994], [-96.81441899976079, 33.093258000026985], [-96.81464200024186, 33.0931900001904], [-96.81478000033411, 33.09314000009991], [-96.81720499974864, 33.09245000016209], [-96.81730400038117, 33.092421999668524], [-96.81782999991262, 33.092272999714304], [-96.81797100037794, 33.09223399988665], [-96.82093700029087, 33.09139800032745], [-96.82213900003994, 33.091058999667815], [-96.82232499981278, 33.09100700017921], [-96.82246499955485, 33.09096799978998], [-96.82265600024888, 33.090913999801344], [-96.82270300010455, 33.09090099991068], [-96.82290999979378, 33.09084299973802], [-96.82329999978113, 33.090737000024326], [-96.82341899960682, 33.09070500034125], [-96.82391200042463, 33.090572999832126], [-96.82417600001529, 33.09050200024601], [-96.82661100037387, 33.08984999983969], [-96.82781599959768, 33.08952400008135], [-96.82844099976165, 33.08935599967389], [-96.83351099959705, 33.08799000029553], [-96.83414499998187, 33.08779299999957], [-96.83491900010877, 33.087553000078074], [-96.83503299991159, 33.08751799995642], [-96.83625599957705, 33.087140000313525], [-96.83686400002259, 33.08698299980309], [-96.83854399962254, 33.08654799964264], [-96.83869299958546, 33.0865090002862], [-96.838892999602, 33.08645799988581], [-96.8389269999372, 33.086446999884096], [-96.83906900022741, 33.08640499965977], [-96.8398709999434, 33.08616700026748], [-96.84135100042513, 33.085728999740226], [-96.84186699991086, 33.08557600010391], [-96.84186600008594, 33.08564899971762], [-96.84186100006306, 33.08593299984479], [-96.84186000023817, 33.08599699966202], [-96.8418569998651, 33.086121000304274], [-96.84185600004021, 33.08618699978577], [-96.84185400039037, 33.086263999996355], [-96.8418490003675, 33.08645600009084], [-96.8424970000977, 33.086275000020905], [-96.8438059999769, 33.085853000007766], [-96.84487499965658, 33.08548900020675], [-96.84548300010212, 33.08532399972421], [-96.84607599958072, 33.085166000199], [-96.8465879997668, 33.08501899981896], [-96.84670599976758, 33.084992999797834], [-96.84719600021236, 33.084858999914495], [-96.84823800012785, 33.08453700014643], [-96.8484380001444, 33.08447100017929], [-96.84889599990379, 33.084317999859685], [-96.85050899955661, 33.08380799988247], [-96.851783000174, 33.083197999788915], [-96.85227500026859, 33.082970000121264], [-96.85249100017866, 33.082870000331425], [-96.8528889996636, 33.082733000028334], [-96.85388500044668, 33.08224599976708], [-96.8555709998944, 33.08141099966482], [-96.85561000025245, 33.08139400005453], [-96.85574999999456, 33.081329999898315], [-96.85572800025324, 33.08120900027553], [-96.85570399996381, 33.08105399963215], [-96.85567100035183, 33.08081200021339], [-96.85563899966645, 33.08056600027114], [-96.85746899995257, 33.07974000016785], [-96.85777500027432, 33.07959800000481], [-96.85744100036348, 33.07896300012854], [-96.85708700036115, 33.07791300001427], [-96.85687899994868, 33.07713700002366], [-96.85655199971053, 33.07662899982605], [-96.85625899980757, 33.07580400003223], [-96.85628499974685, 33.07516699981532], [-96.85717100002672, 33.07275000017698], [-96.85754900031851, 33.07217300027858], [-96.85850400019537, 33.07163199973247], [-96.85923099956827, 33.071171999785086], [-96.85955500033171, 33.07094300011256], [-96.85981899992234, 33.070564999996876], [-96.86001399991602, 33.07002699997931], [-96.8601059999775, 33.06959299998594], [-96.86038399981187, 33.06898000002048], [-96.86072299974558, 33.068571999745764], [-96.86121800021321, 33.068210999886645], [-96.86140599963588, 33.068073000360435], [-96.86153799988037, 33.0679760001439], [-96.86164199963743, 33.06790099994973], [-96.86175299996552, 33.06781999978868], [-96.86195999965474, 33.06770900000455], [-96.86210500031802, 33.067841000200055], [-96.8622380003874, 33.067728999955655], [-96.86206799960978, 33.06756000033868], [-96.86189100005778, 33.06736300020626], [-96.86178300010273, 33.0671720003332], [-96.86166500010197, 33.066966999886006], [-96.86161599969817, 33.06684300009615], [-96.8615270000097, 33.06665400009247], [-96.86146799956016, 33.066463999934584], [-96.86143800032126, 33.066274000119265], [-96.86139899996319, 33.066125000282256], [-96.8613789998717, 33.06589400001093], [-96.86136999965088, 33.06573000017047], [-96.86138899991745, 33.06562299979308], [-96.86140900000892, 33.065424000321286], [-96.86145000001682, 33.06526000035783], [-96.86147299958303, 33.065118999924586], [-96.86153399968241, 33.064959999725666], [-96.86162699956881, 33.06468099998917], [-96.86181800026283, 33.064242000358895], [-96.86240099969572, 33.06279000021192], [-96.86244900027461, 33.06265400004975], [-96.86250000032823, 33.06253400013069], [-96.86257899997095, 33.062319999756035], [-96.86262199962866, 33.06221900003526], [-96.86265099994095, 33.06207800000022], [-96.86270899966729, 33.06186099986801], [-96.86275200022331, 33.06161099967001], [-96.86277099959158, 33.061308999784856], [-96.86276300009396, 33.06095700005193], [-96.86270700001748, 33.0603980000965], [-96.86280999994965, 33.05952000034136], [-96.86316300012703, 33.05862899996767]]]}, "properties": {"ZCTA5": "75024", "AREALAND": 33548920, "AREAWATER": 299035}}, {"type": "Feature", "geometry": {"type": "Polygon", "coordinates": [[[-96.77202899989308, 33.07189299963478], [-96.77203899993881, 33.071725999864405], [-96.77048400001232, 33.07171899978593], [-96.76981199999268, 33.071715000278544], [-96.76913900014814, 33.07171199970695], [-96.76759699974211, 33.07168900008874], [-96.76444500005636, 33.071657999939184], [-96.76339800011799, 33.07165599980775], [-96.76062399982568, 33.071647000344875], [-96.76026300015064, 33.0716480000343], [-96.75996900042277, 33.07163800012829], [-96.75896199976907, 33.07162600008903], [-96.75878199984402, 33.071623999956884], [-96.75860199991897, 33.071623999956884], [-96.75707399975666, 33.071613000358326], [-96.75607899969683, 33.071613000358326], [-96.75477300019068, 33.071604000138265], [-96.75453499964098, 33.071619999692395], [-96.75437900000537, 33.07164000026017], [-96.75417800016392, 33.0716829996964], [-96.75387699986504, 33.071735999760364], [-96.75337999974758, 33.071761999936335], [-96.7530500000347, 33.071773000269054], [-96.7529169999653, 33.07176800032329], [-96.75258399987938, 33.071764999753505], [-96.75225099979345, 33.07176400006541], [-96.7513539996429, 33.07175799967818], [-96.75086400009646, 33.07175499986078], [-96.7503750003749, 33.07175100035504], [-96.75015600009178, 33.07174799978467], [-96.74960600027087, 33.07171599996725], [-96.74906099957452, 33.071626999778694], [-96.74900100019839, 33.071619999692395], [-96.74887399997678, 33.071604000138265], [-96.74852099979937, 33.07158200018199], [-96.7483409998743, 33.07158799982843], [-96.74781600016777, 33.07160200000562], [-96.74713099972932, 33.07158200018199], [-96.74577699964428, 33.071575999782354], [-96.743306000199, 33.07154399990239], [-96.74284400024163, 33.07153300029384], [-96.74238200028428, 33.07151999979661], [-96.74171100008954, 33.071504999915334], [-96.74103900006989, 33.07148799989627], [-96.74039899983731, 33.07147799997207], [-96.73983800014574, 33.07148299993432], [-96.73896899958429, 33.07147200031813], [-96.73842399978624, 33.07146700035527], [-96.73691700043865, 33.07144699974817], [-96.73684200009558, 33.071448999884346], [-96.7367659999276, 33.0714500003288], [-96.73675599988185, 33.07158700013833], [-96.73556099980549, 33.071596000360074], [-96.73529200019198, 33.0715899999614], [-96.73430700017786, 33.07159900018289], [-96.7329220001307, 33.0715779999156], [-96.73201199956137, 33.07157200026848], [-96.7302189999835, 33.07156099991064], [-96.72929000004592, 33.07152999971597], [-96.72735000015494, 33.07150999987604], [-96.72626100038379, 33.071516000280184], [-96.72505299988862, 33.07151400014548], [-96.72253600041259, 33.071516999971095], [-96.72094600032601, 33.071516999971095], [-96.71999599957368, 33.071512000010785], [-96.71915299984983, 33.07150600035916], [-96.71750500003692, 33.07149499999308], [-96.71680000040531, 33.07149799981928], [-96.71578100005603, 33.07149099972271], [-96.71573800039832, 33.071795999865365], [-96.71570599971295, 33.071970000314664], [-96.715673000101, 33.072014999712984], [-96.71557600001661, 33.07225900011458], [-96.71554300040465, 33.0724220001509], [-96.71548899997798, 33.07254999982916], [-96.71545600036602, 33.072785000153445], [-96.71542500040388, 33.07300300012364], [-96.71536000010654, 33.07318500007873], [-96.71534500003794, 33.07323599989519], [-96.7147299999197, 33.073146000066274], [-96.71342600006338, 33.07307799973301], [-96.71269600031742, 33.073051999945925], [-96.7118359999768, 33.07302199995046], [-96.71190899977172, 33.07082500035436], [-96.7114069996314, 33.07080699975214], [-96.71104400030654, 33.07063200031306], [-96.70992000037528, 33.07064599988951], [-96.7098090000472, 33.07068300010376], [-96.7095970003351, 33.07063899972516], [-96.70956399982481, 33.07103000009449], [-96.70953000038796, 33.07141999978711], [-96.70936200015846, 33.07141600026614], [-96.70939800014347, 33.071715000278544], [-96.70937399985402, 33.072080999993275], [-96.70939800014347, 33.072542000137034], [-96.70940300016633, 33.07291399973142], [-96.70923999995972, 33.07298799974247], [-96.70919700030201, 33.073049999846184], [-96.70894799988167, 33.07334199962581], [-96.7087749996293, 33.073432999676136], [-96.708536000153, 33.07351399993961], [-96.70835200003, 33.07355000003273], [-96.70810299960966, 33.07359599979573], [-96.7077889997903, 33.073622000174836], [-96.70774399958447, 33.07361599991433], [-96.7071069997249, 33.07398699982142], [-96.70693099999781, 33.07386199972111], [-96.70655199988114, 33.07353499974482], [-96.70643600042848, 33.07345600034401], [-96.70637199995609, 33.07343100033784], [-96.70628800029048, 33.07342700015551], [-96.70619899970372, 33.07343100033784], [-96.70609999996951, 33.07346800013372], [-96.705971999923, 33.073559000053734], [-96.70585300009732, 33.0737120002696], [-96.7057540003631, 33.07381100026747], [-96.70560600022512, 33.07389799967184], [-96.70526499974325, 33.074030000368886], [-96.70509700041208, 33.07410600026982], [-96.70499799977956, 33.074135000234506], [-96.70491400011399, 33.074135000234506], [-96.7048599996873, 33.07411799997083], [-96.70479100009031, 33.07405300012791], [-96.70473500001383, 33.07400200003227], [-96.70459100007379, 33.073823999673465], [-96.70451900010377, 33.07376600028229], [-96.70435800044528, 33.073647000126726], [-96.70425200014007, 33.07358300035611], [-96.70410800020002, 33.073513000271376], [-96.70396500008489, 33.07345499992235], [-96.70349599955654, 33.073322999868154], [-96.70342600013464, 33.07329299996503], [-96.7032489996843, 33.07321099982651], [-96.70313499988151, 33.07311199990621], [-96.70309999972139, 33.07307499996062], [-96.70307600033027, 33.07303300012564], [-96.70305600023879, 33.072954999948706], [-96.70305100021591, 33.07287899980628], [-96.70306600028454, 33.07282999988766], [-96.70309999972139, 33.07275599974377], [-96.70313499988151, 33.072659999850856], [-96.70315499997298, 33.07252800010942], [-96.70315499997298, 33.07247699988256], [-96.70312999985863, 33.072424999945554], [-96.70309999972139, 33.07238299980023], [-96.70300099998721, 33.072308000351214], [-96.70289300003216, 33.072266000150094], [-96.70284799982632, 33.07226300035001], [-96.7027199997798, 33.07228799993534], [-96.70264100013709, 33.07232500021184], [-96.7025859998855, 33.07236699963207], [-96.70255200044863, 33.072424999945554], [-96.70246299986185, 33.07255300037224], [-96.70241299963314, 33.072603000123706], [-96.70236900015054, 33.0726360000495], [-96.70232899996756, 33.07264300005495], [-96.70227000041633, 33.07264300005495], [-96.70217599980668, 33.0726150000297], [-96.70209700016396, 33.07257799987511], [-96.70204799976017, 33.07252800010942], [-96.701988000384, 33.072511999967624], [-96.70194400000308, 33.07251600019156], [-96.70188499955354, 33.072540999705], [-96.70178099979647, 33.07257799987511], [-96.70162299961271, 33.07271300007915], [-96.70151899985565, 33.0727719998412], [-96.70127199998345, 33.0728760000271], [-96.70143799966478, 33.07124700020244], [-96.70121699973186, 33.06910999962578], [-96.70102000008836, 33.06890499992925], [-96.70005599999067, 33.06876999990503], [-96.69999999991418, 33.06859000005237], [-96.69967400039924, 33.06854999978423], [-96.69960599972889, 33.06867199998071], [-96.69940099968944, 33.068696000110535], [-96.69939900003963, 33.06879999984463], [-96.69873899971556, 33.068804000237314], [-96.69874600028658, 33.06865099976764], [-96.69867799961618, 33.068063000048134], [-96.69852600017853, 33.068085999861374], [-96.69805400017543, 33.06812400012872], [-96.69700100038928, 33.06822400012], [-96.69613900039884, 33.068320000335945], [-96.69591300044304, 33.06834500028868], [-96.69551900025773, 33.06838499989725], [-96.69511099982873, 33.06837300016717], [-96.6947710000701, 33.06834500028868], [-96.69435599996841, 33.068254000245794], [-96.69410300024843, 33.06817400013857], [-96.6940110001869, 33.068146999926014], [-96.69384799998029, 33.06805400021777], [-96.69363600026819, 33.06795300030191], [-96.69340000026665, 33.06783999971456], [-96.6923139999702, 33.06716299965894], [-96.69205200002938, 33.06698799974811], [-96.69160000011777, 33.06670100019505], [-96.69147799991903, 33.06661899993571], [-96.69131899991037, 33.06653400035746], [-96.69105500031972, 33.06640900020856], [-96.69090399980867, 33.06634699994232], [-96.69049299990493, 33.06620499999369], [-96.6902169997204, 33.06613800007095], [-96.69011600033639, 33.066104999966626], [-96.69002099990183, 33.066077000119336], [-96.68931500044532, 33.067754999809985], [-96.68927799973707, 33.06784700010157], [-96.68825199971509, 33.07042099992], [-96.68811000032319, 33.07075000033009], [-96.68753500038792, 33.07217899988473], [-96.68706500003464, 33.07335200009093], [-96.6863079996262, 33.075194999781154], [-96.68622899998346, 33.07536399978893], [-96.68634500033441, 33.07539300009169], [-96.686429, 33.07542199963214], [-96.68663400003942, 33.07548499992468], [-96.68669900033674, 33.07550800005603], [-96.68696499957723, 33.07560300020781], [-96.68704300029334, 33.07567300013513], [-96.68711199989032, 33.075743999649625], [-96.68725400018053, 33.07595500007387], [-96.6874209996868, 33.07623899988869], [-96.68795700016233, 33.07720900024106], [-96.68797999972854, 33.07723999994013], [-96.68801400006373, 33.077268999624465], [-96.68805199969857, 33.077286999657325], [-96.68810199992728, 33.07730200005825], [-96.6881589998287, 33.07730600006439], [-96.68838999980736, 33.07727499963584], [-96.688484000417, 33.07727499963584], [-96.68853700012045, 33.07728899966073], [-96.68855800003685, 33.07730299968347], [-96.68864400025058, 33.07736800013599], [-96.68866900036494, 33.07738499977163], [-96.68871899969534, 33.07739600015403], [-96.68877200029709, 33.07739899977909], [-96.68883800041934, 33.07737899976777], [-96.6892019995691, 33.077259999983], [-96.68925500017086, 33.07725799997893], [-96.68935700027811, 33.07728099964679], [-96.68943199972287, 33.07730600006439], [-96.6894769999287, 33.077342000111294], [-96.68950899971576, 33.07738400014739], [-96.68953499965504, 33.07743000016394], [-96.68958599970864, 33.07763599994284], [-96.68958800025678, 33.07764599991986], [-96.68961500002098, 33.077701999769985], [-96.68963400028755, 33.07772500007449], [-96.68965199983089, 33.07773599966164], [-96.68970000040979, 33.07774100002097], [-96.68973100037192, 33.07773300004807], [-96.68975200028831, 33.077713999733234], [-96.68979599977094, 33.07761300036787], [-96.68981599986242, 33.07756499969719], [-96.68988100015976, 33.07744099978792], [-96.68994700028199, 33.077370999762], [-96.68999599978747, 33.077340000109096], [-96.69005400041209, 33.07732600009225], [-96.69015999981902, 33.07732600009225], [-96.69027000032216, 33.077346999740286], [-96.69036999988128, 33.0773880001498], [-96.69067000035525, 33.07756000008058], [-96.69077599976215, 33.077592000030585], [-96.69083299966356, 33.07761599998546], [-96.69089799996088, 33.07766100025956], [-96.69096599973295, 33.07775099998598], [-96.69107899971085, 33.07793899987019], [-96.69112300009179, 33.07798999994164], [-96.6911599999017, 33.07802399972197], [-96.69119099986384, 33.07805200028389], [-96.69131300006258, 33.07810199992106], [-96.69145899965247, 33.078121999767944], [-96.6916169998362, 33.078097000334985], [-96.6916529998212, 33.07809399998099], [-96.69172200031649, 33.07809399998099], [-96.69181400037802, 33.078121999767944], [-96.69183600011932, 33.078139000010616], [-96.69185299983775, 33.07815099991425], [-96.6918920001958, 33.078195999914655], [-96.69194800027232, 33.07829700011612], [-96.69196699964058, 33.07834499963463], [-96.69199900032592, 33.07842799986931], [-96.69206200007513, 33.07851800031648], [-96.69214800028884, 33.07854700033217], [-96.69219199977147, 33.078543999993585], [-96.692251000221, 33.07851899992782], [-96.69228600038112, 33.07849999978277], [-96.69245700008534, 33.07840099979251], [-96.69253499990315, 33.07836999974984], [-96.69257000006324, 33.0783690001368], [-96.69265700010187, 33.07836399981317], [-96.69285999959315, 33.0783940002452], [-96.69289499975325, 33.078414999638206], [-96.6929190000427, 33.078439000121286], [-96.69294900017992, 33.07850199975861], [-96.69294900017992, 33.07854700033217], [-96.69289399992834, 33.07865999996488], [-96.69283299982897, 33.07873899996154], [-96.69280100004191, 33.07877100023538], [-96.69267700019336, 33.078892999808325], [-96.69265700010187, 33.07895999980414], [-96.6926720001705, 33.07900599974922], [-96.6927009995845, 33.07904799974356], [-96.69274800033847, 33.07909500000015], [-96.69279400036922, 33.079104000206605], [-96.692836000202, 33.079115999978605], [-96.69289499975325, 33.079115999978605], [-96.69298799963967, 33.07910200024441], [-96.69309999979268, 33.07905899991799], [-96.69322000034157, 33.07899799989142], [-96.69336500010652, 33.07892300005992], [-96.69352099974213, 33.07890599996878], [-96.69359000023744, 33.07890700032845], [-96.69364299994088, 33.078919000127314], [-96.6936870003218, 33.07894900037006], [-96.6937209997587, 33.0790339999986], [-96.6937290001546, 33.079125999787315], [-96.69370900006311, 33.079182999804516], [-96.6935950002603, 33.07927199987635], [-96.69349900000083, 33.0793569999455], [-96.69348899995508, 33.07942199963297], [-96.69350800022167, 33.07948500007169], [-96.69353599981076, 33.079517999674145], [-96.69357000014595, 33.079556999872636], [-96.69362200002449, 33.07957399983797], [-96.69369499981941, 33.07958600029856], [-96.6942439998154, 33.079646999917365], [-96.69439399960324, 33.079684000109694], [-96.69443799998417, 33.079723999832304], [-96.6944680001214, 33.07978300016142], [-96.69446100044871, 33.07983599986195], [-96.69443600033435, 33.07988000015719], [-96.69438899958035, 33.07991500030786], [-96.69433400022709, 33.07995600027316], [-96.69412099979176, 33.080153999652815], [-96.69406700026342, 33.08023000028475], [-96.69403700012617, 33.080328999721864], [-96.69403200010329, 33.08039800006202], [-96.69405499966952, 33.0804950001323], [-96.69407899995898, 33.08056800019998], [-96.69413699968528, 33.0806579999629], [-96.69421399967818, 33.080730000309096], [-96.69428900002126, 33.08077000030869], [-96.69441600024287, 33.080807000028685], [-96.69456100000782, 33.080837999965134], [-96.69462399975701, 33.080843999733055], [-96.6946649997649, 33.08083900030284], [-96.69467800018367, 33.080837999965134], [-96.69480100020733, 33.080814999721966], [-96.6949259998808, 33.08074799963375], [-96.69501299991944, 33.080692000217354], [-96.69508499988945, 33.08066500008279], [-96.69522700017968, 33.08062899989043], [-96.69536100007399, 33.0806129997166], [-96.69558399965675, 33.08060699993292], [-96.69571599990124, 33.08063799993992], [-96.69580399976478, 33.08067799998135], [-96.69589199962833, 33.080741000273136], [-96.69592799961335, 33.080785999701234], [-96.69594299968193, 33.080827000013244], [-96.69595699992564, 33.08088800027677], [-96.69595799975055, 33.080935999886776], [-96.69594100003212, 33.08102799994425], [-96.69591900029081, 33.08106599989083], [-96.6958830003058, 33.081157999812326], [-96.6958480001457, 33.08127600001258], [-96.6958270002293, 33.081422999874725], [-96.69582300003134, 33.08149000020147], [-96.69584599959757, 33.08158400033797], [-96.69591900029081, 33.08167399981394], [-96.69594600005499, 33.08169600025725], [-96.6959800003902, 33.081707999675636], [-96.69599999958335, 33.08171600003966], [-96.69605599965985, 33.08171600003966], [-96.6960930003681, 33.08171200023412], [-96.6961569999422, 33.08168700031607], [-96.6961860002545, 33.08167100033471], [-96.69636000033178, 33.081559999960334], [-96.6964110003854, 33.081556000147636], [-96.69647999998239, 33.08156599967901], [-96.6966149997016, 33.08162900008744], [-96.69675399961875, 33.0816879998914], [-96.69692199984826, 33.081774999785274], [-96.69694799978751, 33.08179500030242], [-96.69697399972678, 33.08183200034394], [-96.69699399981829, 33.08185999969365], [-96.69701300008485, 33.08193199980858], [-96.69702999980328, 33.08201999978535], [-96.6970309996282, 33.082096000310266], [-96.69701999975754, 33.08214999969256], [-96.69697700009984, 33.08227100027889], [-96.69697399972678, 33.08229200025172], [-96.69697899974966, 33.08231699999779], [-96.69699399981829, 33.082343000057364], [-96.69703499982616, 33.08237900030082], [-96.69705999994052, 33.08238899973861], [-96.69720100040583, 33.08240000024786], [-96.69727500002567, 33.08243100037561], [-96.69734600017078, 33.08249900025329], [-96.69740999974486, 33.08259399962909], [-96.69741399994282, 33.08260699999524], [-96.69743499985923, 33.08266900009382], [-96.69749000011082, 33.08304599982525], [-96.69749099993574, 33.08306099999675], [-96.69749299958556, 33.08310300031284], [-96.69749000011082, 33.0831429999859], [-96.69747300039239, 33.083185000262816], [-96.6974540001258, 33.0832119996238], [-96.69737299993496, 33.08329499978192], [-96.69725400010928, 33.083499999818685], [-96.69724300023861, 33.083567000068506], [-96.69724599971335, 33.083604000117255], [-96.69726699962975, 33.083634000267345], [-96.69728900026938, 33.08365199975038], [-96.69731100001067, 33.08366300010165], [-96.69735799986634, 33.08367500000436], [-96.69741399994282, 33.08367800016792], [-96.69761200030956, 33.08367800016792], [-96.69764699957133, 33.08367199984068], [-96.6976859999294, 33.08365800007903], [-96.69778700021175, 33.083609000142935], [-96.69786599985446, 33.08358200015121], [-96.69794000037263, 33.083592999758544], [-96.69804499995462, 33.08362200035906], [-96.69810900042701, 33.08365100019737], [-96.69814899971168, 33.083686000352756], [-96.69817999967381, 33.083723000351426], [-96.69821999985679, 33.08380200031674], [-96.69825800038993, 33.08394199986186], [-96.6982559998418, 33.08400100037218], [-96.6982419995981, 33.08404899978864], [-96.69819499974243, 33.084141999900694], [-96.69815799993252, 33.0841830001535], [-96.6981219999475, 33.08420699981536], [-96.69806800041914, 33.08422699977945], [-96.69801200034264, 33.084238000058804], [-96.6979880000532, 33.084238000058804], [-96.69786699967936, 33.084241000203214], [-96.69781599962576, 33.08424900033683], [-96.69778700021175, 33.08425999986075], [-96.69776000044757, 33.08427699966976], [-96.69771000021883, 33.08432499968826], [-96.69766199963995, 33.08438399994164], [-96.69764800029458, 33.08442500008168], [-96.69764699957133, 33.084459999929095], [-96.69765400014235, 33.08450700030436], [-96.69766599983791, 33.084547000091554], [-96.69771600006663, 33.084608000032986], [-96.69776000044757, 33.0846370002988], [-96.69779499970936, 33.08464799977423], [-96.69785300033398, 33.084644999643764], [-96.69798399985524, 33.08461999980675], [-96.6981360001912, 33.08457500008228], [-96.69822300022983, 33.08455000022536], [-96.69846599990407, 33.08447699969939], [-96.69860700036939, 33.08444300015546], [-96.69880300018797, 33.084415000122696], [-96.69887199978494, 33.084411999984276], [-96.69890199992217, 33.08441699996402], [-96.6989329998843, 33.08442899976385], [-96.69898399993795, 33.084469000339205], [-96.699015999725, 33.08451599995698], [-96.69905199971002, 33.08464200026581], [-96.69907500017455, 33.08475299976109], [-96.69910500031176, 33.08486800028381], [-96.69917300008383, 33.08508800024298], [-96.69931100017608, 33.085415999707614], [-96.69936399987954, 33.08552100028278], [-96.69939599966659, 33.085566000275854], [-96.69944099987242, 33.08560699986482], [-96.6994879997281, 33.08564199999443], [-96.69952399971312, 33.08566199963217], [-96.69957599959164, 33.08567599982748], [-96.69991299987552, 33.085715000306024], [-96.70040400014521, 33.08584699982888], [-96.7005589999559, 33.085896000023304], [-96.7006640004362, 33.085942999631165], [-96.70072999966014, 33.085994000330174], [-96.70076499982024, 33.08603299991507], [-96.70081300039912, 33.08610799970496], [-96.70085400040702, 33.08621199992997], [-96.70092100035417, 33.0863390003419], [-96.70092300000398, 33.086361999893136], [-96.70091100030842, 33.086412000074894], [-96.70091299995825, 33.086457000364696], [-96.70091999963093, 33.08647999988506], [-96.70094999976817, 33.08651400014672], [-96.70098300027843, 33.08653400033882], [-96.70102099991327, 33.08654799964264], [-96.70107699998978, 33.08655099970819], [-96.70123699982334, 33.08652500014006], [-96.70132999970976, 33.086515999940374], [-96.70136099967189, 33.086522000073565], [-96.70140800042589, 33.08654700012254], [-96.70143400036514, 33.086581999878014], [-96.70144299968766, 33.086630000143735], [-96.70143900038802, 33.08665500016195], [-96.70142900034229, 33.086680000173054], [-96.70140199967979, 33.086714000357446], [-96.70123199980048, 33.08683799998867], [-96.70120000001343, 33.086904999953376], [-96.70119499999053, 33.08694699993895], [-96.70120000001343, 33.08703399989894], [-96.7012059998612, 33.087063000117375], [-96.70122500012778, 33.087103000247225], [-96.70126500031074, 33.087157999831625], [-96.70130799996845, 33.0871809999213], [-96.70136499986985, 33.08719799991826], [-96.70137900011356, 33.08719699965289], [-96.70140000002996, 33.087196000140175], [-96.70143699983988, 33.08718700000911], [-96.70158400015298, 33.08711400016688], [-96.7018190003296, 33.087077000089636], [-96.70188600027676, 33.08708000013714], [-96.70196699956931, 33.08710800007403], [-96.70200900030041, 33.08713499973594], [-96.70204500028542, 33.08717600009865], [-96.70208100027044, 33.087251999931105], [-96.70218599985243, 33.087548999785334], [-96.70229499963237, 33.08777499985886], [-96.70235599973174, 33.0878599999893], [-96.70237800037135, 33.08788199963779], [-96.70245299981612, 33.08792899968944], [-96.70247700010556, 33.087940000258364], [-96.70256699961892, 33.08794699979863], [-96.70263000026644, 33.08793499972626], [-96.70266899972619, 33.08792200014773], [-96.70270999973405, 33.087908000309994], [-96.70282799973485, 33.08785099992625], [-96.70285000037447, 33.08784999966832], [-96.70289999970487, 33.08785399994735], [-96.70292999984208, 33.087861999752285], [-96.70295099975849, 33.087870999814264], [-96.7029859999186, 33.087902000271335], [-96.70307399978213, 33.08800200036038], [-96.70311499979, 33.08803300001868], [-96.70314900012521, 33.088040000304204], [-96.7031709998665, 33.08803900004841], [-96.7032550004304, 33.088019999702034], [-96.70331300015674, 33.08799200005555], [-96.70333800027109, 33.087984000262445], [-96.70337099988305, 33.08798900003917], [-96.70346499959437, 33.08804800009221], [-96.703487000234, 33.08805100010665], [-96.70351200034837, 33.08805100010665], [-96.70357699974737, 33.08801799994259], [-96.70360500023479, 33.08801399967102], [-96.70369100044852, 33.08801499992708], [-96.70382600016772, 33.08803900004841], [-96.70388700026707, 33.088057000135095], [-96.70392699955174, 33.08808499976088], [-96.70396900028285, 33.08814099973831], [-96.70397500013064, 33.08816400032352], [-96.70398500017639, 33.08820599970775], [-96.70398799965112, 33.08824000005471], [-96.70400199989481, 33.08826700012162], [-96.70402899965902, 33.08828800016797], [-96.70405499959826, 33.08829799968649], [-96.70419400041376, 33.08831899972544], [-96.70432699958482, 33.08839099982083], [-96.70445700017949, 33.0884449998537], [-96.70461799983798, 33.08851000035101], [-96.70481299983163, 33.08856900005453], [-96.70488899999962, 33.08857200005113], [-96.70493199965732, 33.088549000325294], [-96.70496799964234, 33.088495999854246], [-96.70493599985528, 33.08839699982613], [-96.70493300038055, 33.088357999784435], [-96.7049369996802, 33.088331999997926], [-96.70496400034268, 33.08828999992123], [-96.70502500044206, 33.088256999846934], [-96.70513900024488, 33.08818899990561], [-96.70527099959105, 33.08814300024755], [-96.70535400033003, 33.08813299995876], [-96.7054220001021, 33.08813299995876], [-96.70555299962335, 33.088160000058586], [-96.70564700023299, 33.08818400014034], [-96.70570900015727, 33.08823899980119], [-96.70573500009652, 33.08827399963586], [-96.70575099999007, 33.08830600020368], [-96.70578299977713, 33.08841199983756], [-96.7058039996935, 33.08858599978314], [-96.70582499960989, 33.08863900019989], [-96.70584100040175, 33.08865299992123], [-96.7058940001052, 33.08867900036547], [-96.70596900044826, 33.08870099980904], [-96.70603300002237, 33.08870099980904], [-96.70607999987803, 33.08869300008044], [-96.70628699956727, 33.08863399970754], [-96.7063300001233, 33.08863399970754], [-96.70636699993321, 33.08864399993935], [-96.70642600038275, 33.08866999963365], [-96.70655299970606, 33.08875100016612], [-96.70669599982118, 33.08881299969632], [-96.70674400040008, 33.08882199966089], [-96.7068029999513, 33.08882199966089], [-96.70688199959402, 33.088807999966455], [-96.70699899976988, 33.08874599968009], [-96.70706800026517, 33.08869700032119], [-96.70710000005222, 33.08867900036547], [-96.70715299975568, 33.08866600014433], [-96.7072120002052, 33.08866600014433], [-96.70725999988579, 33.088675000123956], [-96.70730699974146, 33.08869700032119], [-96.70737100021388, 33.08873699970773], [-96.70747699962078, 33.0888890003718], [-96.70751400032901, 33.08892399994773], [-96.70757799990311, 33.08895500003362], [-96.70771599999537, 33.0890040000014], [-96.70793399955525, 33.08905799965781], [-96.70796600024063, 33.08907000032962], [-96.70804599970825, 33.089125000188886], [-96.70814100014282, 33.08916900035265], [-96.70818399980051, 33.08917700003794], [-96.70823700040228, 33.08917700003794], [-96.70835899970271, 33.08915999967094], [-96.70850799966563, 33.08910100036398], [-96.70862999986436, 33.08906199988195], [-96.70870600003232, 33.08905499967779], [-96.70874700004022, 33.089055999922024], [-96.7087949997208, 33.08906700034995], [-96.70887099988879, 33.08914000007612], [-96.70903400009539, 33.089330000182485], [-96.70911299973811, 33.089421999737894], [-96.70916799998969, 33.08948000009529], [-96.70923800030991, 33.08954300010272], [-96.70979300015367, 33.08863399970754], [-96.71089799981839, 33.08673099969203], [-96.71092499958256, 33.08668600029511], [-96.71107500026868, 33.08637799977373], [-96.71137499984434, 33.085762000010625], [-96.71155099957143, 33.08540899996588], [-96.7116250000896, 33.08519200016899], [-96.71182500010615, 33.084678999862234], [-96.71187799980959, 33.0845149996611], [-96.712266999972, 33.084606999738156], [-96.71277800033319, 33.08475299976109], [-96.7131239999396, 33.08489900029426], [-96.71354300023924, 33.085109000295844], [-96.7139909999529, 33.085375000029586], [-96.71413800026599, 33.08545599983397], [-96.71449700029119, 33.08561100024591], [-96.714687000262, 33.08568300030065], [-96.71497799961682, 33.085773000097966], [-96.7152449995805, 33.08583800031248], [-96.71550599969642, 33.08588900031977], [-96.71570499988803, 33.08591800016315], [-96.71615799962458, 33.08594599971737], [-96.71760700014417, 33.08592499986437], [-96.7186089998767, 33.08591800016315], [-96.71896600025207, 33.08591800016315], [-96.72001099964231, 33.085934999651606], [-96.7211439997944, 33.08595699978169], [-96.72194800005853, 33.08594599971737], [-96.72228000031956, 33.08595900034062], [-96.72291600035419, 33.085962999953075], [-96.72355300021373, 33.08596600003863], [-96.72408899979092, 33.08596600003863], [-96.72447600030354, 33.0859679998447], [-96.72777899960205, 33.08599100024563], [-96.72946300029827, 33.085951999889495], [-96.72993799977611, 33.08597299973604], [-96.73025999999139, 33.0859679998447], [-96.7310270004456, 33.08598800016092], [-96.73179400000147, 33.086010999803996], [-96.7347599999144, 33.08605000013396], [-96.73511899993959, 33.086056000298996], [-96.73657500013188, 33.08602199986019], [-96.73657199975884, 33.08616799979188], [-96.73655099984245, 33.08722099999748], [-96.73647300002465, 33.087516999694685], [-96.7363719997423, 33.08777199983506], [-96.73622799980225, 33.08807800023159], [-96.73604299985433, 33.0883780003097], [-96.73586399975417, 33.088627999718405], [-96.73574400010358, 33.08875899988945], [-96.73551700032287, 33.08898300012611], [-96.73517099981814, 33.0892840003687], [-96.73486600021961, 33.08953400021195], [-96.7347880004018, 33.08959299998101], [-96.73448499955478, 33.08984599965146], [-96.7342529997512, 33.09006400023828], [-96.73397300026704, 33.09039399966322], [-96.73380300038772, 33.090665999818064], [-96.73370100028046, 33.0908599997796], [-96.73360400019608, 33.09107900033138], [-96.73354300009672, 33.09131299985414], [-96.73349100021818, 33.091665000066676], [-96.73344199981439, 33.09314500033604], [-96.73344199981439, 33.093282999665874], [-96.73344199981439, 33.09361100017055], [-96.73341900024816, 33.095259000026], [-96.73340400017955, 33.09686799981239], [-96.733420999898, 33.098767999762764], [-96.73340099980649, 33.09957199975635], [-96.73338499991299, 33.10019499991117], [-96.73335200030101, 33.100825000185246], [-96.73335200030101, 33.10098699984833], [-96.73336700036963, 33.10245899971434], [-96.73335099957778, 33.10349000019954], [-96.73334800010305, 33.10386999967801], [-96.73335499977574, 33.10585200014375], [-96.73335599960066, 33.10594499976728], [-96.73335499977574, 33.10600299975779], [-96.73334599955491, 33.108620999760134], [-96.73334599955491, 33.10900399967565], [-96.73334599955491, 33.11039999986906], [-96.73334399990509, 33.11081700019774], [-96.73333899988222, 33.11257100006898], [-96.7333380000573, 33.11360499993235], [-96.7333029998972, 33.115650999797644], [-96.7333029998972, 33.11577699987098], [-96.73330200007229, 33.11701200003882], [-96.73348700002022, 33.117025999755406], [-96.73348600019531, 33.11723800034971], [-96.733492999868, 33.11769899999652], [-96.73349000039327, 33.11797600024449], [-96.73345199986012, 33.11843200023027], [-96.73343199976863, 33.11850999994123], [-96.73341099985223, 33.118585999766545], [-96.73321900023163, 33.119109999742484], [-96.73300099977342, 33.11963000001656], [-96.73236700028691, 33.120953000302514], [-96.73219900005743, 33.12140600025057], [-96.73208899955425, 33.12194000015808], [-96.73203199965286, 33.12244199966292], [-96.73203599985082, 33.12294499991841], [-96.73210799982083, 33.123389999791584], [-96.73217499976798, 33.123679999704024], [-96.73223199966938, 33.12381499968683], [-96.73236200026403, 33.1241500002523], [-96.73257700034921, 33.12461400000014], [-96.73264500012125, 33.12475999977368], [-96.73276300012203, 33.125012999769936], [-96.73289899966616, 33.12530099992519], [-96.73293899984913, 33.125283999775135], [-96.73310299988067, 33.12523099964003], [-96.73321300038384, 33.12518999972645], [-96.73342500009596, 33.12511000026208], [-96.73425799977409, 33.12482800000785], [-96.73545200002553, 33.12442699971412], [-96.73602000028811, 33.12423700025071], [-96.7372370001058, 33.1238639997797], [-96.73771800032974, 33.12368800000972], [-96.7392869996016, 33.12315900023188], [-96.74025400007235, 33.122844999712676], [-96.74049400027185, 33.12275299978394], [-96.7405679998917, 33.12272200033418], [-96.74063599966377, 33.122691000121115], [-96.74165099981508, 33.122359000356425], [-96.74289500029526, 33.12195200010178], [-96.74486500032343, 33.12127499983643], [-96.7457699999716, 33.120942999985], [-96.74684100019941, 33.12058000017726], [-96.74753300031053, 33.12035099977342], [-96.74831499993505, 33.12010200020404], [-96.74871199959507, 33.119976000316136], [-96.74958600017939, 33.11968700010533], [-96.75025700037413, 33.11942200027641], [-96.75053900040645, 33.11931100017004], [-96.75068599982123, 33.11925299993667], [-96.75078100025576, 33.119215999845665], [-96.75090299955619, 33.1191820001956], [-96.75104499984643, 33.11913000004149], [-96.75175899969885, 33.11889600009551], [-96.7524309997185, 33.118666999814906], [-96.75323499998262, 33.11839499979334], [-96.75845700015401, 33.11634799995623], [-96.7599140001712, 33.115736000058064], [-96.76394799993889, 33.11413799990933], [-96.76467199983708, 33.113830000017806], [-96.76787200010172, 33.11247000011247], [-96.76802000023974, 33.112402000091166], [-96.76802400043769, 33.112081000364576], [-96.76803000028548, 33.1115360000857], [-96.76805700004965, 33.1101890000818], [-96.76807700014115, 33.10921800009591], [-96.76813800024051, 33.10804100036391], [-96.7681790002484, 33.10723300019403], [-96.76818300044634, 33.107158999652505], [-96.76818500009618, 33.107121999734616], [-96.7682429998225, 33.10696999974566], [-96.76827600033278, 33.106893999652584], [-96.76834999995262, 33.10675499982715], [-96.76843600016635, 33.106622000048226], [-96.76861899956613, 33.10627799991817], [-96.76871999984847, 33.10613699977107], [-96.76885899976564, 33.10600299975779], [-96.76902600017021, 33.105861999921906], [-96.76921899961575, 33.10568900036383], [-96.76942299983025, 33.105563000118984], [-96.76996000013068, 33.105219999665955], [-96.77021799987355, 33.10504700034889], [-96.77035799961563, 33.10494499995692], [-96.77049499988297, 33.10483300030495], [-96.77070999996815, 33.10462700027673], [-96.77081899974809, 33.10451099994603], [-96.77095199981747, 33.10434300033727], [-96.77107700038927, 33.10416799990169], [-96.77118600016921, 33.10397299988724], [-96.77123899987266, 33.10388700021035], [-96.77128800027646, 33.10377199973214], [-96.77134900037582, 33.10359799974322], [-96.771368999569, 33.10354099974974], [-96.77140000042945, 33.1033919998299], [-96.77142499964549, 33.103194000101055], [-96.7714530001329, 33.10256599968195], [-96.77145499978273, 33.10244299975019], [-96.7714749998742, 33.101373000239064], [-96.7714749998742, 33.10124599972059], [-96.77148699956976, 33.101166000271526], [-96.77148099972199, 33.101077999890435], [-96.77148200044522, 33.101004000185846], [-96.77149900016366, 33.10052699991372], [-96.7714949999657, 33.09988200020418], [-96.77150400018651, 33.099411999995624], [-96.77150699966126, 33.099102000147695], [-96.77152099990496, 33.09783500022765], [-96.77152699975274, 33.096910000306686], [-96.77152299955478, 33.09630200036794], [-96.77151900025513, 33.09620999973363], [-96.77152000008005, 33.0957050003367], [-96.77152099990496, 33.09531200018306], [-96.77154600001931, 33.09387399995736], [-96.77154400036949, 33.093306999854455], [-96.7715579997149, 33.092788999973855], [-96.7715800003545, 33.09125900007746], [-96.77162399983712, 33.08941200034714], [-96.77163100040814, 33.08825399983958], [-96.77163600043099, 33.08748799962598], [-96.77162800003507, 33.08610599990204], [-96.77162800003507, 33.08594899980345], [-96.77163999973064, 33.08458100034791], [-96.77166900004296, 33.082906999920695], [-96.771672000416, 33.082646999893896], [-96.77168699958631, 33.0804120003426], [-96.7717069996778, 33.07980000008306], [-96.77172400029454, 33.07929600012559], [-96.77174699986077, 33.079070000090994], [-96.77176500030244, 33.07893899978844], [-96.7717900004168, 33.07883399963497], [-96.77181100033319, 33.078663000299514], [-96.77180999960996, 33.07859800005117], [-96.77184899996801, 33.07767700021767], [-96.77188199957999, 33.07681499983582], [-96.77193799965647, 33.075764000032606], [-96.77199299990806, 33.07453799976671], [-96.77201600037259, 33.07373500011175], [-96.77202899989308, 33.07189299963478]]]}, "properties": {"ZCTA5": "75025", "AREALAND": 24744023, "AREAWATER": 130945}}, {"type": "Feature", "geometry": {"type": "Polygon", "coordinates": [[[-96.77203899993881, 33.071725999864405], [-96.77202899989308, 33.07146199963933], [-96.77201100034974, 33.0713300003589], [-96.77197600018962, 33.07116600011028], [-96.77185399999088, 33.07077800019605], [-96.77177600017309, 33.0705920002203], [-96.77170599985288, 33.070466000110464], [-96.77145700033086, 33.07009399995234], [-96.77108599971179, 33.069553000173734], [-96.7710389998561, 33.06948500007479], [-96.77071699964084, 33.06899999968272], [-96.77057900044689, 33.06871700031286], [-96.77053300041614, 33.06860999980325], [-96.77049700043112, 33.068511999700775], [-96.77043299995871, 33.06829500037605], [-96.7704050003696, 33.06818500016702], [-96.77038699992795, 33.068060999835], [-96.77036000016378, 33.06768399987107], [-96.7703559999658, 33.06748300031651], [-96.77035900033886, 33.06733799997449], [-96.77037500023238, 33.065798000159766], [-96.7703769998822, 33.06529099975007], [-96.77039000030099, 33.064332000260904], [-96.77039500032387, 33.064097999722], [-96.77039899962352, 33.06347000012924], [-96.77039399960064, 33.06245099995921], [-96.77039199995082, 33.061924999882834], [-96.77038799975286, 33.06159399999815], [-96.7703969999737, 33.061053999723114], [-96.77039899962352, 33.06058600016794], [-96.7704050003696, 33.05921999971936], [-96.7704110002174, 33.057774999981326], [-96.77039500032387, 33.05673099990494], [-96.77037100003442, 33.05601199980133], [-96.7703690003846, 33.05578799993382], [-96.7703839995549, 33.054233000003315], [-96.77039100012591, 33.053944999627674], [-96.77037199985934, 33.05351700026763], [-96.77036099998868, 33.053352999954505], [-96.77034400027024, 33.052867000363705], [-96.77032800037672, 33.051771999886064], [-96.7703409998972, 33.05091599979582], [-96.77035000011801, 33.050004000297285], [-96.77035099994293, 33.04924999993994], [-96.77036099998868, 33.04843699968016], [-96.77039100012591, 33.045614000326566], [-96.77037500023238, 33.04543999973122], [-96.77032699965349, 33.04508799980404], [-96.77021899969847, 33.04418400020265], [-96.77008999982702, 33.04314900024534], [-96.76990800025216, 33.04171900001187], [-96.7698839999627, 33.04143500025666], [-96.7698839999627, 33.0413029996239], [-96.7696370000905, 33.0413099998621], [-96.76940799976164, 33.04130000027468], [-96.76923899970724, 33.04126599986172], [-96.76886999963631, 33.04115699980041], [-96.76863399963474, 33.04111599983223], [-96.76851900000702, 33.04110300011383], [-96.76839599998337, 33.04109399978636], [-96.76765700001658, 33.04106400019413], [-96.76759199971924, 33.04105499986262], [-96.76531700009254, 33.04105499986262], [-96.76508399956572, 33.04105399982574], [-96.76485199976213, 33.041051999751915], [-96.76194500029875, 33.04101299980963], [-96.75936799964994, 33.040992999813206], [-96.75874699968392, 33.04099900003854], [-96.75821699995448, 33.040992999813206], [-96.75714800027482, 33.04099599992592], [-96.75559800037121, 33.04098799962515], [-96.75451299989967, 33.04098200015212], [-96.75338499977045, 33.04097599992563], [-96.75338200029573, 33.0411199999769], [-96.75322499993689, 33.04111899994076], [-96.75203099968543, 33.0411199999769], [-96.75102700030307, 33.0411199999769], [-96.75023700028265, 33.0411199999769], [-96.74940600025434, 33.0411199999769], [-96.74841500039247, 33.041139999944484], [-96.74788800013778, 33.041145000123905], [-96.74770300018984, 33.04113699983667], [-96.74726700017176, 33.04113699983667], [-96.74567999955991, 33.041128000265665], [-96.74382199968471, 33.04112700022963], [-96.74381599983694, 33.041040000061166], [-96.74380900016423, 33.0409539998448], [-96.7396599998705, 33.04097299981229], [-96.73935299972383, 33.04096700033824], [-96.73873900032883, 33.04094800036947], [-96.73836199986198, 33.04095599992086], [-96.73769900006316, 33.04093699994971], [-96.73672999994258, 33.04093199975848], [-96.73591999983067, 33.04092700031996], [-96.73462699984502, 33.04092300016653], [-96.73349000039327, 33.04092000005133], [-96.73332100033888, 33.04092000005133], [-96.73276499977186, 33.04093099972015], [-96.73220900010317, 33.04094000006433], [-96.73182599978854, 33.04094800036947], [-96.73144400019711, 33.0409539998448], [-96.73100900000392, 33.04096200014867], [-96.7301760003258, 33.04095900003479], [-96.72893500021868, 33.0409539998448], [-96.72647399992081, 33.04096400022455], [-96.7255519996559, 33.04097299981229], [-96.72399599990452, 33.04099199977563], [-96.72370899984934, 33.04098400022755], [-96.72343000019008, 33.04098400022755], [-96.72315200035574, 33.040985000265216], [-96.72283699981314, 33.04099599992592], [-96.72262400027613, 33.040992999813206], [-96.72240599981792, 33.04098799962515], [-96.72053200004922, 33.040950999730654], [-96.71955399970781, 33.04094000006433], [-96.71894199996264, 33.04093399983498], [-96.71832900039253, 33.04092899964358], [-96.7173910002341, 33.040911999743656], [-96.71681799994866, 33.040902000111075], [-96.71624600038643, 33.04089099968557], [-96.71623499961748, 33.04107099969826], [-96.71605899989038, 33.04106700030443], [-96.71486700018706, 33.041051999751915], [-96.71399000012799, 33.0410410000982], [-96.71369899987486, 33.04103799998701], [-96.71309100032762, 33.04108300013783], [-96.71292499974795, 33.04108900035705], [-96.71225699992627, 33.04111099965116], [-96.71171399977806, 33.0411060002228], [-96.71114499969057, 33.04110000000476], [-96.71057099958021, 33.04109199971344], [-96.71013000043757, 33.04108300013783], [-96.7099560003603, 33.04106899962481], [-96.70980700039739, 33.04105799997322], [-96.70963599979484, 33.04103299980152], [-96.70943099975543, 33.040992999813206], [-96.7092020003249, 33.04094400021699], [-96.70902499987456, 33.040900000033794], [-96.70871400042826, 33.04079599971884], [-96.70850799966563, 33.04069800027998], [-96.70831699986991, 33.04058800022668], [-96.7081129996554, 33.040413000201085], [-96.70797000043859, 33.04026100013338], [-96.70770299957657, 33.03983099968078], [-96.70750500010818, 33.03946399986022], [-96.70713799968706, 33.03878599991113], [-96.70702399988424, 33.038574999996364], [-96.70691799957903, 33.038425999969796], [-96.70670199966895, 33.03822100036043], [-96.70655299970606, 33.03810000020549], [-96.70649099978176, 33.0380519998241], [-96.70645399997184, 33.038034000050914], [-96.70603700022032, 33.03784900024409], [-96.7059300000902, 33.03781600007896], [-96.7057269997006, 33.0377780002843], [-96.7047540002804, 33.03764700027194], [-96.7045569997386, 33.037618999657624], [-96.70439899955485, 33.03760099979597], [-96.70426300001073, 33.037580999778], [-96.70343299980733, 33.039129999702176], [-96.70305100021591, 33.03987499964486], [-96.70254900007559, 33.040837999875286], [-96.7024720000827, 33.040969999698795], [-96.70219300042343, 33.04149699999498], [-96.70197400014032, 33.041903999975055], [-96.69951500039059, 33.046491000232145], [-96.69943000000178, 33.046667000203435], [-96.69869400040803, 33.04802899981108], [-96.69622899991221, 33.05268800026914], [-96.69600200013147, 33.0531060001414], [-96.69589500000137, 33.053305000355984], [-96.69561200014415, 33.05386599995864], [-96.69485300008587, 33.05528699984463], [-96.69473799955983, 33.05550000012724], [-96.69443700015925, 33.056039999838774], [-96.69387599956939, 33.05710100037048], [-96.69380799979731, 33.057222999873794], [-96.69370999988803, 33.05737000001959], [-96.6936510003368, 33.05747699988943], [-96.69324799993068, 33.058310000086095], [-96.69295500002771, 33.05892100016764], [-96.6927859999733, 33.05927199993055], [-96.69265399972882, 33.059590000264095], [-96.69239699981088, 33.06020200031535], [-96.69077800031027, 33.06422200036819], [-96.6903280000485, 33.06534000010278], [-96.6901520003214, 33.06577300023625], [-96.69007699997832, 33.065954000335225], [-96.69002099990183, 33.066077000119336], [-96.69011600033639, 33.066104999966626], [-96.6902169997204, 33.06613800007095], [-96.69049299990493, 33.06620499999369], [-96.69090399980867, 33.06634699994232], [-96.69105500031972, 33.06640900020856], [-96.69131899991037, 33.06653400035746], [-96.69147799991903, 33.06661899993571], [-96.69160000011777, 33.06670100019505], [-96.69205200002938, 33.06698799974811], [-96.6923139999702, 33.06716299965894], [-96.69340000026665, 33.06783999971456], [-96.69363600026819, 33.06795300030191], [-96.69384799998029, 33.06805400021777], [-96.6940110001869, 33.068146999926014], [-96.69410300024843, 33.06817400013857], [-96.69435599996841, 33.068254000245794], [-96.6947710000701, 33.06834500028868], [-96.69511099982873, 33.06837300016717], [-96.69551900025773, 33.06838499989725], [-96.69591300044304, 33.06834500028868], [-96.69613900039884, 33.068320000335945], [-96.69700100038928, 33.06822400012], [-96.69805400017543, 33.06812400012872], [-96.69852600017853, 33.068085999861374], [-96.69867799961618, 33.068063000048134], [-96.69874600028658, 33.06865099976764], [-96.69873899971556, 33.068804000237314], [-96.69939900003963, 33.06879999984463], [-96.69940099968944, 33.068696000110535], [-96.69960599972889, 33.06867199998071], [-96.69967400039924, 33.06854999978423], [-96.69999999991418, 33.06859000005237], [-96.70005599999067, 33.06876999990503], [-96.70102000008836, 33.06890499992925], [-96.70121699973186, 33.06910999962578], [-96.70143799966478, 33.07124700020244], [-96.70127199998345, 33.0728760000271], [-96.70151899985565, 33.0727719998412], [-96.70162299961271, 33.07271300007915], [-96.70178099979647, 33.07257799987511], [-96.70188499955354, 33.072540999705], [-96.70194400000308, 33.07251600019156], [-96.701988000384, 33.072511999967624], [-96.70204799976017, 33.07252800010942], [-96.70209700016396, 33.07257799987511], [-96.70217599980668, 33.0726150000297], [-96.70227000041633, 33.07264300005495], [-96.70232899996756, 33.07264300005495], [-96.70236900015054, 33.0726360000495], [-96.70241299963314, 33.072603000123706], [-96.70246299986185, 33.07255300037224], [-96.70255200044863, 33.072424999945554], [-96.7025859998855, 33.07236699963207], [-96.70264100013709, 33.07232500021184], [-96.7027199997798, 33.07228799993534], [-96.70284799982632, 33.07226300035001], [-96.70289300003216, 33.072266000150094], [-96.70300099998721, 33.072308000351214], [-96.70309999972139, 33.07238299980023], [-96.70312999985863, 33.072424999945554], [-96.70315499997298, 33.07247699988256], [-96.70315499997298, 33.07252800010942], [-96.70313499988151, 33.072659999850856], [-96.70309999972139, 33.07275599974377], [-96.70306600028454, 33.07282999988766], [-96.70305100021591, 33.07287899980628], [-96.70305600023879, 33.072954999948706], [-96.70307600033027, 33.07303300012564], [-96.70309999972139, 33.07307499996062], [-96.70313499988151, 33.07311199990621], [-96.7032489996843, 33.07321099982651], [-96.70342600013464, 33.07329299996503], [-96.70349599955654, 33.073322999868154], [-96.70396500008489, 33.07345499992235], [-96.70410800020002, 33.073513000271376], [-96.70425200014007, 33.07358300035611], [-96.70435800044528, 33.073647000126726], [-96.70451900010377, 33.07376600028229], [-96.70459100007379, 33.073823999673465], [-96.70473500001383, 33.07400200003227], [-96.70479100009031, 33.07405300012791], [-96.7048599996873, 33.07411799997083], [-96.70491400011399, 33.074135000234506], [-96.70499799977956, 33.074135000234506], [-96.70509700041208, 33.07410600026982], [-96.70526499974325, 33.074030000368886], [-96.70560600022512, 33.07389799967184], [-96.7057540003631, 33.07381100026747], [-96.70585300009732, 33.0737120002696], [-96.705971999923, 33.073559000053734], [-96.70609999996951, 33.07346800013372], [-96.70619899970372, 33.07343100033784], [-96.70628800029048, 33.07342700015551], [-96.70637199995609, 33.07343100033784], [-96.70643600042848, 33.07345600034401], [-96.70655199988114, 33.07353499974482], [-96.70693099999781, 33.07386199972111], [-96.7071069997249, 33.07398699982142], [-96.70774399958447, 33.07361599991433], [-96.7077889997903, 33.073622000174836], [-96.70810299960966, 33.07359599979573], [-96.70835200003, 33.07355000003273], [-96.708536000153, 33.07351399993961], [-96.7087749996293, 33.073432999676136], [-96.70894799988167, 33.07334199962581], [-96.70919700030201, 33.073049999846184], [-96.70923999995972, 33.07298799974247], [-96.70940300016633, 33.07291399973142], [-96.70939800014347, 33.072542000137034], [-96.70937399985402, 33.072080999993275], [-96.70939800014347, 33.071715000278544], [-96.70936200015846, 33.07141600026614], [-96.70953000038796, 33.07141999978711], [-96.70956399982481, 33.07103000009449], [-96.7095970003351, 33.07063899972516], [-96.7098090000472, 33.07068300010376], [-96.70992000037528, 33.07064599988951], [-96.71104400030654, 33.07063200031306], [-96.7114069996314, 33.07080699975214], [-96.71190899977172, 33.07082500035436], [-96.7118359999768, 33.07302199995046], [-96.71269600031742, 33.073051999945925], [-96.71342600006338, 33.07307799973301], [-96.7147299999197, 33.073146000066274], [-96.71534500003794, 33.07323599989519], [-96.71536000010654, 33.07318500007873], [-96.71542500040388, 33.07300300012364], [-96.71545600036602, 33.072785000153445], [-96.71548899997798, 33.07254999982916], [-96.71554300040465, 33.0724220001509], [-96.71557600001661, 33.07225900011458], [-96.715673000101, 33.072014999712984], [-96.71570599971295, 33.071970000314664], [-96.71573800039832, 33.071795999865365], [-96.71578100005603, 33.07149099972271], [-96.71680000040531, 33.07149799981928], [-96.71750500003692, 33.07149499999308], [-96.71915299984983, 33.07150600035916], [-96.71999599957368, 33.071512000010785], [-96.72094600032601, 33.071516999971095], [-96.72253600041259, 33.071516999971095], [-96.72505299988862, 33.07151400014548], [-96.72626100038379, 33.071516000280184], [-96.72735000015494, 33.07150999987604], [-96.72929000004592, 33.07152999971597], [-96.7302189999835, 33.07156099991064], [-96.73201199956137, 33.07157200026848], [-96.7329220001307, 33.0715779999156], [-96.73430700017786, 33.07159900018289], [-96.73529200019198, 33.0715899999614], [-96.73556099980549, 33.071596000360074], [-96.73675599988185, 33.07158700013833], [-96.7367659999276, 33.0714500003288], [-96.73684200009558, 33.071448999884346], [-96.73691700043865, 33.07144699974817], [-96.73842399978624, 33.07146700035527], [-96.73896899958429, 33.07147200031813], [-96.73983800014574, 33.07148299993432], [-96.74039899983731, 33.07147799997207], [-96.74103900006989, 33.07148799989627], [-96.74171100008954, 33.071504999915334], [-96.74238200028428, 33.07151999979661], [-96.74284400024163, 33.07153300029384], [-96.743306000199, 33.07154399990239], [-96.74577699964428, 33.071575999782354], [-96.74713099972932, 33.07158200018199], [-96.74781600016777, 33.07160200000562], [-96.7483409998743, 33.07158799982843], [-96.74852099979937, 33.07158200018199], [-96.74887399997678, 33.071604000138265], [-96.74900100019839, 33.071619999692395], [-96.74906099957452, 33.071626999778694], [-96.74960600027087, 33.07171599996725], [-96.75015600009178, 33.07174799978467], [-96.7503750003749, 33.07175100035504], [-96.75086400009646, 33.07175499986078], [-96.7513539996429, 33.07175799967818], [-96.75225099979345, 33.07176400006541], [-96.75258399987938, 33.071764999753505], [-96.7529169999653, 33.07176800032329], [-96.7530500000347, 33.071773000269054], [-96.75337999974758, 33.071761999936335], [-96.75387699986504, 33.071735999760364], [-96.75417800016392, 33.0716829996964], [-96.75437900000537, 33.07164000026017], [-96.75453499964098, 33.071619999692395], [-96.75477300019068, 33.071604000138265], [-96.75607899969683, 33.071613000358326], [-96.75707399975666, 33.071613000358326], [-96.75860199991897, 33.071623999956884], [-96.75878199984402, 33.071623999956884], [-96.75896199976907, 33.07162600008903], [-96.75996900042277, 33.07163800012829], [-96.76026300015064, 33.0716480000343], [-96.76062399982568, 33.071647000344875], [-96.76339800011799, 33.07165599980775], [-96.76444500005636, 33.071657999939184], [-96.76759699974211, 33.07168900008874], [-96.76913900014814, 33.07171199970695], [-96.76981199999268, 33.071715000278544], [-96.77048400001232, 33.07171899978593], [-96.77203899993881, 33.071725999864405]]]}, "properties": {"ZCTA5": "75023", "AREALAND": 23824659, "AREAWATER": 15985}}]}'
SCRIPT_SNAPSHOT = '"""Clip Zenodo\'s 2021 Texas canopy raster; use --download on a cloud computer."""\nimport argparse\nimport hashlib\nimport json\nimport zipfile\nfrom datetime import datetime, timezone\nfrom pathlib import Path\n\nimport geopandas as gpd\nimport matplotlib\nmatplotlib.use(\'Agg\')\nimport matplotlib.pyplot as plt\nimport numpy as np\nimport rasterio\nfrom rasterio.mask import mask\nfrom rasterio.plot import plotting_extent\nimport requests\n\nROOT = Path(__file__).resolve().parents[1]\nURL = \'https://zenodo.org/records/8200761/files/TX_10m_canopy_cover_2021.zip?download=1\'\nMD5 = \'79aec57e7339c8aa74e8a424b0000d32\'\nCOUNTY = \'https://tigerweb.geo.census.gov/arcgis/rest/services/TIGERweb/State_County/MapServer/1/query\'\n\n\ndef download_source(folder):\n    folder.mkdir(parents=True, exist_ok=True)\n    archive = folder / \'TX_10m_canopy_cover_2021.zip\'\n    if not archive.exists():\n        partial = archive.with_suffix(\'.zip.part\')\n        with requests.get(URL, stream=True, timeout=(30, 180)) as response:\n            response.raise_for_status()\n            with partial.open(\'wb\') as stream:\n                total = 0\n                for chunk in response.iter_content(8 * 1024 * 1024):\n                    stream.write(chunk)\n                    total += len(chunk)\n                    print(f\'Downloaded {total / 1e6:.0f} MB\', flush=True)\n        partial.replace(archive)\n    digest = hashlib.md5()\n    with archive.open(\'rb\') as stream:\n        for chunk in iter(lambda: stream.read(8 * 1024 * 1024), b\'\'):\n            digest.update(chunk)\n    if digest.hexdigest() != MD5:\n        raise ValueError(f\'Checksum mismatch: delete {archive} and retry\')\n    with zipfile.ZipFile(archive) as bundle:\n        for member in bundle.infolist():\n            target = (folder / member.filename).resolve()\n            if not target.is_relative_to(folder.resolve()):\n                raise ValueError(\'Unsafe archive member\')\n            if not target.exists() or target.stat().st_size != member.file_size:\n                bundle.extract(member, folder)\n    return folder / \'TX_10m_canopy_cover_2021.img\'\n\n\ndef prepare_scope(boundaries, zipcodes, full_zctas, output):\n    scope = gpd.read_file(boundaries).to_crs(4326)\n    scope = scope[scope.ZCTA5.astype(str).isin(zipcodes)].copy()\n    if set(scope.ZCTA5.astype(str)) != set(zipcodes):\n        raise ValueError(\'Some requested ZCTAs are missing from the boundary file\')\n    if not full_zctas:\n        response = requests.get(COUNTY, params={\n            \'where\': "GEOID=\'48085\'", \'outFields\': \'GEOID,NAME\',\n            \'returnGeometry\': \'true\', \'outSR\': \'4326\', \'f\': \'geojson\',\n        }, timeout=90)\n        response.raise_for_status()\n        payload = response.json()\n        if len(payload.get(\'features\', [])) != 1:\n            raise ValueError(f\'Could not retrieve Collin County: {payload}\')\n        (output / \'collin_county_source.geojson\').write_text(json.dumps(payload))\n        county = gpd.GeoDataFrame.from_features(payload[\'features\'], crs=4326)\n        scope.geometry = scope.geometry.intersection(county.geometry.iloc[0])\n        if scope.geometry.is_empty.any():\n            raise ValueError(\'A requested ZCTA has no area in Collin County\')\n    scope.to_file(output / \'clipping_boundaries.geojson\', driver=\'GeoJSON\')\n    return scope\n\n\ndef clip_one(src, geometries, target):\n    # crop=True reads only the bounding window, not the statewide pixel array.\n    data, transform = mask(src, geometries, crop=True, filled=False, indexes=1)\n    scaled = data.astype(\'float32\') * src.scales[0] + src.offsets[0]\n    valid = scaled.compressed()\n    if not valid.size:\n        raise ValueError(\'No valid canopy pixels intersect this scope\')\n    if valid.min() < 0 or valid.max() > 100:\n        raise ValueError(\'Values are outside 0–100: inspect source class key/NoData before mapping\')\n    profile = dict(driver=\'GTiff\', height=data.shape[0], width=data.shape[1], count=1,\n                   dtype=\'float32\', crs=src.crs, transform=transform, nodata=-9999,\n                   compress=\'deflate\', tiled=True)\n    with rasterio.open(target, \'w\', **profile) as dst:\n        dst.write(scaled.filled(-9999), 1)\n        dst.update_tags(source=URL, vintage=\'2021\', units=\'percent canopy cover\',\n                        boundary_rule=\'pixel center inside polygon; outside and source NoData masked\')\n    return scaled, transform\n\n\ndef main():\n    p = argparse.ArgumentParser(description=__doc__)\n    p.add_argument(\'--boundaries\', type=Path, default=ROOT / \'output/scope_screen/zcta_boundaries.geojson\')\n    p.add_argument(\'--zipcodes\', nargs=\'+\', default=[\'75025\', \'75024\', \'75074\', \'75023\'])\n    p.add_argument(\'--full-zctas\', action=\'store_true\', help=\'Keep portions outside Collin County too\')\n    p.add_argument(\'--source-raster\', type=Path, help=\'Existing .img with companion .ige in same folder\')\n    p.add_argument(\'--download\', action=\'store_true\', help=\'Download statewide archive (~2.25 GB); intended for Colab\')\n    p.add_argument(\'--cache\', type=Path, default=ROOT / \'data/raw/texas_canopy_2021\')\n    p.add_argument(\'--output\', type=Path, default=ROOT / \'output/canopy_2021\')\n    args = p.parse_args()\n    if not args.source_raster and not args.download:\n        p.error(\'Provide --source-raster, or run with --download on Colab\')\n    args.output.mkdir(parents=True, exist_ok=True)\n    source = args.source_raster or download_source(args.cache)\n    scope = prepare_scope(args.boundaries, args.zipcodes, args.full_zctas, args.output)\n    with rasterio.open(source) as src:\n        if src.crs is None or src.count != 1:\n            raise ValueError(\'Expected a georeferenced, single-band canopy raster\')\n        projected = scope.to_crs(src.crs)\n        image, transform = clip_one(src, projected.geometry.tolist(), args.output / \'canopy_scope_2021.tif\')\n        for _, row in projected.iterrows():\n            clip_one(src, [row.geometry], args.output / f\'canopy_{row.ZCTA5}_2021.tif\')\n        fig, ax = plt.subplots(figsize=(10, 9))\n        plot = ax.imshow(image, extent=plotting_extent(image, transform), cmap=\'YlGn\', vmin=0, vmax=100)\n        projected.boundary.plot(ax=ax, color=\'#333333\', linewidth=.8)\n        for _, row in projected.iterrows():\n            point = row.geometry.representative_point()\n            ax.text(point.x, point.y, row.ZCTA5, fontsize=10, ha=\'center\')\n        fig.colorbar(plot, ax=ax, label=\'Canopy cover (%)\', shrink=.7)\n        ax.set_title(\'2021 canopy cover · 10 m Texas dataset\\n\' + (\'Full ZCTAs\' if args.full_zctas else \'ZCTAs within Collin County\'))\n        ax.set_axis_off()\n        fig.text(.1, .02, \'Diamond, Sunde & Elliott · Zenodo 8200761\\nCanopy cover is not time-specific pedestrian shade.\', fontsize=9)\n        fig.savefig(args.output / \'canopy_preview.png\', dpi=180, bbox_inches=\'tight\')\n        plt.close(fig)\n        metadata = dict(source=URL, archive_md5=MD5, vintage=2021,\n                        processed_at_utc=datetime.now(timezone.utc).isoformat(),\n                        zipcodes=args.zipcodes, collin_county_only=not args.full_zctas,\n                        source_crs=str(src.crs), source_resolution=list(src.res),\n                        source_nodata=src.nodata, source_scale=src.scales[0], source_offset=src.offsets[0],\n                        boundary_source=str(args.boundaries), county_source=COUNTY if not args.full_zctas else None,\n                        note=\'ZCTAs approximate postal ZIPs. Pixel-center mask; source NoData preserved. No resampling.\')\n        (args.output / \'source_metadata.json\').write_text(json.dumps(metadata, indent=2) + \'\\n\')\n    print(f\'Clipped outputs: {args.output.resolve()}\')\n\n\nif __name__ == \'__main__\':\n    main()\n'
(WORK / 'zcta_boundaries.geojson').write_text(BOUNDARY_SNAPSHOT)
(WORK / 'clip_texas_canopy.py').write_text(SCRIPT_SNAPSHOT)


In [ ]:
import subprocess, sys
command = [sys.executable, str(WORK / 'clip_texas_canopy.py'),
           '--download', '--boundaries', str(WORK / 'zcta_boundaries.geojson'),
           '--cache', str(WORK / 'statewide_cache'), '--output', str(WORK / 'results'),
           '--zipcodes', *ZIPCODES]
if FULL_ZCTAS:
    command.append('--full-zctas')
subprocess.run(command, check=True)


In [ ]:
from IPython.display import display, Image
display(Image(filename=str(WORK / 'results/canopy_preview.png')))
print((WORK / 'results/source_metadata.json').read_text())


In [ ]:
# Bundle ONLY the small clipped outputs, never the statewide cache.
from google.colab import files
bundle = shutil.make_archive(str(WORK / 'empath_canopy_2021_clipped'), 'zip', WORK / 'results')
print('Download size (MB):', round(Path(bundle).stat().st_size / 1e6, 2))
files.download(bundle)


After downloading, extract the small ZIP into `EmPath/output/canopy_2021/`. Open `canopy_preview.png` to view it; use `canopy_scope_2021.tif` for analysis or the per-ZCTA GeoTIFFs for individual areas. Keep `source_metadata.json` and `clipping_boundaries.geojson` with the outputs. If a cell fails, retain its error message; a checksum mismatch requires deleting the cached archive and rerunning the download cell.